# 🔬 From One Click to a 3D Structure

## Interactive CryoET Tomogram Segmentation with Prompt Propagation

---

## Part 0 — Tutorial Overview

### 0.1 Why Do We Need Interactive CryoET Segmentation?

Cryogenic electron tomography, or **CryoET**, reconstructs a three-dimensional view of a frozen biological sample from electron microscopy images acquired at different tilt angles. The reconstructed 3D image is called a **tomogram**.

A tomogram can be represented as a three-dimensional array:

$$V\in\mathbb{R}^{D\times H\times W},$$

where $D$ is the number of slices, while $H$ and $W$ are the height and width of each slice.

CryoET tomograms can reveal membranes, ribosomes, vesicles, macromolecular complexes, and other structures inside biological samples. However, automatic analysis is difficult because CryoET data often contain low signal-to-noise ratios, weak boundaries, reconstruction artefacts, direction-dependent blur, and multiple nearby structures with similar appearances.

Before a structure can be measured or analyzed, it must be separated from the surrounding background. This process is called **segmentation**.

In a fully manual workflow, a researcher may need to inspect many slices and draw the target boundary separately on every slice. This process is slow, repetitive, and sensitive to human inconsistency.

This tutorial investigates a more interactive question:

> Can a user identify a target on one 2D slice and let the system propagate that information through neighboring slices to recover the complete 3D structure?

---

### 0.2 The Central Workflow

The user first selects an informative slice from a 3D tomogram and provides a spatial prompt, such as a point placed inside the target.

The point prompt can be represented as:

$$P_{z_0}=(x_0,y_0),$$

where $z_0$ is the selected slice index and $(x_0,y_0)$ is the user-selected location inside the target.

A prompt-guided segmentation method then predicts an initial 2D mask:

$$\hat{M}_{z_0}=f(I_{z_0},P_{z_0}),$$

where $I_{z_0}$ is the selected slice, $P_{z_0}$ is the user prompt, and $\hat{M}_{z_0}$ is the predicted target mask.

The current mask is then used to automatically generate a prompt for an adjacent slice:

$$\hat{M}_{z}\rightarrow P_{z+1}\rightarrow\hat{M}_{z+1}.$$

This process is repeated in both directions from the selected seed slice:

$$z_0\rightarrow z_0+1\rightarrow z_0+2\rightarrow\cdots$$

$$z_0\rightarrow z_0-1\rightarrow z_0-2\rightarrow\cdots$$

Finally, all accepted 2D masks are placed at their corresponding slice positions and combined into a 3D mask:

$$\hat{M}_{3D}=\{\hat{M}_{z_{\min}},\ldots,\hat{M}_{z_0},\ldots,\hat{M}_{z_{\max}}\}.$$

The complete tutorial workflow is:

$$\boxed{\text{3D tomogram}\rightarrow\text{user prompt}\rightarrow\text{initial 2D mask}\rightarrow\text{bidirectional propagation}\rightarrow\text{3D mask}\rightarrow\text{measurement}}$$

---

### 0.3 Research Papers Behind This Tutorial

This tutorial is based primarily on two research papers: **CryoSAM** and **Segment Anything**.

| Reference paper | Knowledge used in this tutorial |
|---|---|
| **CryoSAM: Training-free CryoET Tomogram Segmentation with Foundation Models** | CryoET segmentation, cross-plane self-prompting, bidirectional slice propagation, automatic prompt generation, and the planned similar-target search |
| **Segment Anything** | Point prompts, box prompts, mask prompts, prompt-guided 2D segmentation, and interactive mask correction |

These papers play different roles in the tutorial. Segment Anything provides the basic 2D promptable segmentation model, while CryoSAM provides the strategy for extending a 2D prompt and mask across a 3D CryoET tomogram.

#### Primary Reference — CryoSAM

The main reference for this tutorial is:

> Zhao, Y., Bian, H., Mu, M., Uddin, M. R., Li, Z., Li, X., Wang, T., and Xu, M. **CryoSAM: Training-free CryoET Tomogram Segmentation with Foundation Models.** MICCAI 2024.

CryoSAM addresses the problem of segmenting structures in a 3D CryoET tomogram without training a new segmentation model for every target class.

The paper introduces two main components:

1. **Cross-Plane Self-Prompting:** A mask predicted on one slice is converted into a prompt for an adjacent slice. Repeating this process allows a single user prompt to develop into a 3D segmentation.
2. **Hierarchical Feature Matching:** Features extracted from one segmented target are used to search for additional structures with similar appearances in the full tomogram.

The main pipeline implemented in this tutorial is based on the first component:

$$\text{user prompt}\rightarrow\text{initial 2D mask}\rightarrow\text{automatic prompt generation}\rightarrow\text{bidirectional propagation}\rightarrow\text{3D mask}.$$

The second component will be retained as a planned extension:

$$\text{segmented target}\rightarrow\text{feature extraction}\rightarrow\text{similar target search}.$$

#### Foundation Reference — Segment Anything

The second core reference is:

> Kirillov, A. et al. **Segment Anything.** ICCV 2023.

The Segment Anything Model, or SAM, is a promptable 2D segmentation foundation model. It receives an image together with a spatial prompt and predicts the mask of the requested object:

$$\text{image}+\text{point, box, or mask prompt}\rightarrow\text{predicted 2D mask}.$$

A positive point indicates a location inside the target, while a negative point indicates an area that should be excluded.

SAM is designed primarily for 2D images and does not automatically understand how one target extends through a 3D tomogram. CryoSAM addresses this limitation by propagating segmentation information between neighboring slices and viewing planes.

The relationship between the two papers can therefore be summarized as:

$$\boxed{\text{SAM provides prompt-guided 2D segmentation, while CryoSAM extends the interaction into a 3D CryoET workflow.}}$$

#### How This Tutorial Adapts the Papers

This notebook is a lightweight educational implementation rather than a complete reproduction of either paper.

The tutorial preserves the following research ideas:

- selecting a target with a spatial prompt;
- generating an initial 2D mask;
- deriving a new prompt from the previous mask;
- propagating segmentation in both slice directions;
- stopping propagation when the prediction becomes unreliable;
- combining 2D masks into a 3D mask;
- allowing the user to inspect and correct the result.

The tutorial simplifies or omits:

- the original large-scale SAM training process;
- complete CryoSAM cross-plane implementation details;
- full-tomogram hierarchical feature matching;
- large-scale comparisons across multiple CryoET datasets;
- reproduction of every experiment reported in the papers.

The purpose is to help learners understand and interact with the central method while maintaining a complete and runnable notebook.

---

### 0.4 What Makes This Workflow Interactive?

The system does not generate a final result without user involvement. Instead, the user can inspect and influence several stages of the segmentation process.

During this tutorial, the user will be able to:

- browse through the slices of a 3D tomogram;
- choose an informative starting slice;
- place a point prompt inside a target;
- inspect the initial 2D prediction;
- adjust segmentation sensitivity;
- control the propagation stopping criteria;
- inspect the propagated masks slice by slice;
- identify and correct an inaccurate slice;
- update the final 3D mask;
- compare measurements before and after correction.

This produces a human-in-the-loop workflow:

$$\text{user input}\rightarrow\text{model prediction}\rightarrow\text{user inspection}\rightarrow\text{correction}.$$

The purpose is not to remove the user from the analysis. Instead, the system reduces repetitive manual annotation while keeping the user involved in important decisions.

---

### 0.5 Tutorial Data Strategy

This notebook will use two types of data.

#### Stage 1 — Synthetic 3D Volume

A small synthetic volume will first be used to explain:

- how a 3D object appears across multiple 2D slices;
- how a 2D mask represents one cross-section of a target;
- how multiple 2D masks form one 3D mask;
- how propagation parameters affect the final result.

The synthetic volume has a known ground-truth mask, allowing us to calculate Dice, Intersection over Union, volume error, and centroid error.

#### Stage 2 — Real CryoET Tomogram Crop

After the basic concepts are established, the same workflow will be applied to a cropped real CryoET tomogram.

The real-data section will focus on:

- selecting a target in noisy experimental data;
- generating an initial prompt-guided mask;
- propagating the mask across neighboring slices;
- inspecting segmentation continuity;
- correcting propagation errors;
- measuring the final 3D structure.

Using a small crop keeps the notebook suitable for Google Colab while preserving the appearance and difficulty of real CryoET data.

---

### 0.6 Learning Objectives

By the end of this tutorial, you will be able to:

1. Explain how a 3D tomogram is represented as a sequence of 2D slices.
2. Distinguish between a 2D image, a 2D mask, and a 3D mask.
3. Use a point prompt to select a target structure on one slice.
4. Generate and interpret an initial 2D segmentation prediction.
5. Extract an automatic prompt from a predicted mask.
6. Propagate segmentation results to adjacent slices.
7. Perform forward and backward propagation from a selected seed slice.
8. Define stopping criteria based on mask area, overlap, and position.
9. Combine multiple 2D masks into a complete 3D mask.
10. Inspect the 3D mask from the XY, XZ, and YZ directions.
11. Correct an inaccurate slice using an additional user prompt.
12. Calculate the volume, centroid, bounding box, and slice span of a segmented structure.
13. Explain how prompt placement and propagation thresholds affect the final result.
14. Identify important limitations of prompt-based CryoET segmentation.

---

### 0.7 Tutorial Roadmap

| Part | Topic | Main outcome |
|---|---|---|
| **Part 0** | Tutorial overview | Understand the problem, research basis, and complete workflow |
| **Part 1** | Tomograms, slices, voxels, and masks | Understand how 2D slices form a 3D structure |
| **Part 2** | Data loading and exploration | Load and browse a real CryoET tomogram crop |
| **Part 3** | User prompt and initial segmentation | Generate the initial 2D target mask |
| **Part 4** | Automatic prompt generation | Convert one mask into a prompt for the next slice |
| **Part 5** | Bidirectional propagation | Recover the target across neighboring slices |
| **Part 6** | 3D mask construction and correction | Build, inspect, and refine the 3D result |
| **Part 7** | Quantitative measurement | Measure the segmented 3D structure |
| **Part 8** | Stress tests and failure modes | Study prompt and propagation sensitivity |
| **Part 9** | Planned extension | Extract features and search for similar targets |

---

### 0.8 Expected Final Outputs

At the end of the main tutorial pipeline, you will produce:

- an interactively selected seed slice;
- a user-defined point prompt;
- an initial predicted 2D mask;
- forward- and backward-propagated masks;
- a combined 3D segmentation mask;
- XY, XZ, and YZ views of the segmented target;
- a slice-by-slice segmentation explorer;
- a cross-sectional area curve;
- a 3D measurement summary;
- an evaluation of how user choices affect the final segmentation.

The final result will not be only a segmentation image. It will be a complete transformation from a user-selected target to a measurable 3D structure.

---

### 0.9 Scope and Limitations

This notebook is designed to make the central interaction and propagation ideas observable and modifiable. It is not intended to reproduce every component of the original CryoSAM or Segment Anything systems.

The current notebook will not:

- train or fine-tune SAM;
- reproduce the complete CryoSAM feature-matching pipeline;
- process every tomogram from a large CryoET dataset;
- guarantee biologically correct segmentation without expert review;
- replace specialist CryoET annotation software;
- produce validated clinical or biological conclusions.

The first version will prioritize a complete and understandable pipeline:

$$\text{prompt}\rightarrow\text{2D prediction}\rightarrow\text{propagation}\rightarrow\text{3D result}\rightarrow\text{measurement}.$$

After this pipeline is complete, a future extension will add:

$$\text{segmented target}\rightarrow\text{feature extraction}\rightarrow\text{similar target search}.$$

---

### 0.10 Tutorial Information

| Item | Description |
|---|---|
| **Level** | Intermediate |
| **Platform** | Google Colab |
| **Hardware** | CPU for introductory sections; a free-tier T4 GPU is recommended for foundation-model inference |
| **Core tools** | Python, NumPy, SciPy, Matplotlib, PyTorch, and a SAM-compatible prompt interface |
| **Data** | Synthetic 3D volume followed by a cropped real CryoET tomogram |
| **Main interaction** | Slice selection, point prompting, propagation control, inspection, and correction |
| **Primary reference** | CryoSAM |
| **Foundation reference** | Segment Anything |

> **Important:** This notebook is intended for research and educational purposes only. It should not be used for clinical diagnosis or unvalidated biological conclusions.

---

### 0.11 Key References

1. Zhao, Y., Bian, H., Mu, M., Uddin, M. R., Li, Z., Li, X., Wang, T., and Xu, M. **CryoSAM: Training-free CryoET Tomogram Segmentation with Foundation Models.** *Medical Image Computing and Computer Assisted Intervention (MICCAI)*, 2024. [Paper](https://papers.miccai.org/miccai-2024/paper/0532_paper.pdf)

2. Kirillov, A. et al. **Segment Anything.** *IEEE/CVF International Conference on Computer Vision (ICCV)*, 2023. [Paper](https://arxiv.org/abs/2304.02643) · [Official Repository](https://github.com/facebookresearch/segment-anything)

# Part 1 — Understanding Tomograms, Slices, Voxels, and Masks

Before working with a real CryoET tomogram, we first need to understand how a three-dimensional image is represented and how a 3D segmentation is assembled from multiple 2D masks.

## 1.1 A Tomogram Is a 3D Volume

A grayscale 2D image is normally represented as a matrix:

$$I\in\mathbb{R}^{H\times W},$$

where each pixel contains an intensity value.

A CryoET tomogram contains one additional spatial dimension:

$$V\in\mathbb{R}^{D\times H\times W},$$

where $D$ is the number of slices, $H$ is the height, and $W$ is the width.

A location in a 2D image is called a **pixel**. A location in a 3D volume is called a **voxel**. A voxel can be imagined as a very small cube containing one intensity value.

## 1.2 Viewing a 3D Volume as 2D Slices

A 3D tomogram cannot be displayed completely on a normal 2D screen. Instead, we inspect it one slice at a time.

The three common viewing planes are:

| Plane | Fixed coordinate | Extracted slice |
|---|---|---|
| **XY** | $z$ | $V[z,:,:]$ |
| **XZ** | $y$ | $V[:,y,:]$ |
| **YZ** | $x$ | $V[:,:,x]$ |

The XY view is usually treated as the default slice direction. However, the XZ and YZ views are also important because some structures may appear clearer in one direction than another.

## 1.3 One 3D Object Appears Across Multiple Slices

Consider a spherical object inside a 3D volume. Near the bottom of the sphere, its 2D cross-section appears as a small circle. The circle becomes larger near the center of the sphere and becomes smaller again near the top.

Therefore, the different circles seen across neighboring slices are not separate objects. They are different cross-sections of the same 3D object.

## 1.4 What Is a Mask?

A segmentation mask has the same spatial dimensions as the image being segmented. A binary 2D mask contains two possible values:

$$M_z(y,x)=\begin{cases}1,&\text{if the pixel belongs to the target}\\0,&\text{if the pixel belongs to the background}\end{cases}$$

A 2D mask identifies the target on one slice, while a 3D mask identifies the target throughout the complete volume:

$$M_{3D}\in\{0,1\}^{D\times H\times W}.$$

Each value of 1 in a 3D mask represents a voxel that belongs to the segmented structure.

## 1.5 From 2D Masks to a 3D Mask

If a target appears from slice $z_{\min}$ to slice $z_{\max}$, we can segment it separately on each slice:

$$M_{z_{\min}},M_{z_{\min}+1},\ldots,M_{z_{\max}}.$$

Placing every 2D mask back at its original slice position produces a 3D mask:

$$M_{3D}[z,:,:]=M_z.$$

The 3D result is therefore not created by stretching one 2D mask. It is created by stacking multiple target cross-sections in the correct spatial order.

## 1.6 Why Start with Synthetic Data?

Real CryoET tomograms are noisy and visually complex. In this part, we will use a simple synthetic volume containing one spherical target.

The synthetic example allows us to observe the relationship between a 3D object, its 2D cross-sections, and its masks without being distracted by imaging noise or ambiguous biological structures.

The red overlay used in this part represents the known ground-truth mask. In later parts, the mask will instead be predicted from a user prompt.

## Task 1 — Explore a 3D Volume Slice by Slice

### Task Goal

The goal of this task is to understand how one 3D object changes its appearance across different 2D slices.

You will generate a small synthetic volume containing one spherical target and inspect it from the XY, XZ, and YZ directions.

### What You Need to Do

1. Run the synthetic-volume generation cell.
2. Inspect the volume shape and identify the meaning of each dimension.
3. Use the interactive controls to change the viewing plane and slice index.
4. Observe when the target first appears, reaches its largest cross-section, and disappears.
5. Compare the grayscale image with the red ground-truth mask overlay.
6. Record your observations in the final Task 1 answer cell.

### Questions to Consider

- Why is the target small near its first visible slice?
- On which slice does the target have its largest cross-sectional area?
- Is the largest cross-section located near the center of the 3D object?
- How does changing from the XY plane to the XZ or YZ plane affect its appearance?
- Why can the same 3D object produce differently sized 2D masks?

### Expected Output

The interactive viewer will display:

- the selected grayscale slice;
- the corresponding binary mask;
- the target mask as a red overlay on the original slice;
- the current plane and slice index.

Your final Task 1 result should include:

- the first visible XY slice;
- the XY slice with the largest target area;
- the last visible XY slice;
- one short explanation of how the 2D cross-sections form a 3D object.

In [ ]:
%pip install -q ipywidgets

In [ ]:
# Import the packages used to create and visualize the synthetic 3D volume.
import numpy as np
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter

# Use a fixed random seed so that every user initially obtains the same volume.
SEED = 42
rng = np.random.default_rng(SEED)

# Create a cubic volume so that the XY, XZ, and YZ planes have the same slice range.
VOLUME_SHAPE = (64, 64, 64)
CENTER = (32, 32, 32)
RADIUS = 18

# Generate the z, y, and x coordinate of every voxel.
z, y, x = np.indices(VOLUME_SHAPE)

# Define the ground-truth target as a sphere centered inside the volume.
distance_squared = (z - CENTER[0]) ** 2 + (y - CENTER[1]) ** 2 + (x - CENTER[2]) ** 2
ground_truth_mask = distance_squared <= RADIUS ** 2

# Create a grayscale volume with background noise and a brighter target region.
volume = 0.20 + 0.05 * rng.normal(size=VOLUME_SHAPE)
volume = volume + 0.65 * ground_truth_mask.astype(np.float32)

# Apply slight smoothing so that the synthetic object has softer image boundaries.
volume = gaussian_filter(volume, sigma=0.8)

# Normalize the complete volume to the range [0, 1].
volume = (volume - volume.min()) / (volume.max() - volume.min())

# Print the most important properties before beginning the interactive exploration.
print(f"Volume shape: {volume.shape}")
print(f"Ground-truth mask shape: {ground_truth_mask.shape}")
print(f"Number of target voxels: {ground_truth_mask.sum():,}")
print("Dimension order: (z, y, x)")

In [ ]:
def extract_slice(array, plane, slice_index):
    """Extract one 2D slice from a 3D array using the selected viewing plane."""
    if plane == "XY":
        return array[slice_index, :, :]
    if plane == "XZ":
        return array[:, slice_index, :]
    if plane == "YZ":
        return array[:, :, slice_index]
    raise ValueError("Plane must be 'XY', 'XZ', or 'YZ'.")

def show_volume_slice(plane="XY", slice_index=32):
    """Display the image, binary mask, and mask overlay for one selected slice."""
    image_slice = extract_slice(volume, plane, slice_index)
    mask_slice = extract_slice(ground_truth_mask, plane, slice_index)

    # Display the original slice, the binary mask, and the overlay side by side.
    fig, axes = plt.subplots(1, 3, figsize=(14, 4))

    axes[0].imshow(image_slice, cmap="gray", origin="lower")
    axes[0].set_title(f"{plane} image, slice {slice_index}")
    axes[0].axis("off")

    axes[1].imshow(mask_slice, cmap="gray", origin="lower", vmin=0, vmax=1)
    axes[1].set_title(f"{plane} ground-truth mask")
    axes[1].axis("off")

    axes[2].imshow(image_slice, cmap="gray", origin="lower")
    axes[2].imshow(np.ma.masked_where(~mask_slice, mask_slice), cmap="Reds", alpha=0.45, origin="lower")
    axes[2].set_title(f"Image + mask overlay")
    axes[2].axis("off")

    plt.tight_layout()
    plt.show()

# Choose an initial plane and slice before opening the interactive viewer.
INITIAL_PLANE = "XY"  # WriteYourCodeHere: Change this to "XZ" or "YZ" to start from another plane.
INITIAL_SLICE = 32  # WriteYourCodeHere: Change this value to begin from another slice.

show_volume_slice(INITIAL_PLANE, INITIAL_SLICE)

In [ ]:
# Try to load ipywidgets so that the plane and slice can be changed interactively.
try:
    import ipywidgets as widgets
    from IPython.display import display
    widgets_available = True
except ImportError:
    widgets_available = False
    print("ipywidgets is not available. Change INITIAL_PLANE and INITIAL_SLICE in the previous cell instead.")

if widgets_available:
    # Use the plane menu to inspect the object from three orthogonal directions.
    plane_control = widgets.Dropdown(options=["XY", "XZ", "YZ"], value=INITIAL_PLANE, description="Plane:")

    # Move this slider to observe how the target cross-section changes across the volume.
    slice_control = widgets.IntSlider(value=INITIAL_SLICE, min=0, max=VOLUME_SHAPE[0] - 1, step=1, description="Slice:", continuous_update=False)

    # Update the visualization whenever the user changes the plane or slice index.
    interactive_viewer = widgets.interactive_output(show_volume_slice, {"plane": plane_control, "slice_index": slice_control})

    display(widgets.HBox([plane_control, slice_control]), interactive_viewer)

In [ ]:
# Count the number of target pixels on every XY slice.
xy_cross_sectional_area = ground_truth_mask.sum(axis=(1, 2))

# Plot how the target cross-sectional area changes along the z direction.
plt.figure(figsize=(8, 4))
plt.plot(np.arange(VOLUME_SHAPE[0]), xy_cross_sectional_area, color="darkred", linewidth=2)
plt.xlabel("XY slice index, z")
plt.ylabel("Target area in pixels")
plt.title("Target Cross-Sectional Area Across XY Slices")
plt.grid(alpha=0.3)
plt.show()

# Use the curve together with the slice explorer before completing the answer cell below.

In [ ]:
# Enter the slice indices you identified using the interactive viewer and area curve.
FIRST_VISIBLE_SLICE = None  # WriteYourCodeHere: Replace None with the first XY slice containing the target.
LARGEST_AREA_SLICE = None  # WriteYourCodeHere: Replace None with the XY slice containing the largest target area.
LAST_VISIBLE_SLICE = None  # WriteYourCodeHere: Replace None with the last XY slice containing the target.

# Write one short sentence explaining how multiple 2D cross-sections form one 3D object.
YOUR_EXPLANATION = ""  # WriteYourCodeHere: Replace the empty string with your explanation.

# Check whether all required Task 1 responses have been completed.
student_answers = [FIRST_VISIBLE_SLICE, LARGEST_AREA_SLICE, LAST_VISIBLE_SLICE]

if any(answer is None for answer in student_answers):
    print("Task incomplete: fill in all three slice indices marked with WriteYourCodeHere.")
elif not YOUR_EXPLANATION.strip():
    print("Task incomplete: add your explanation in YOUR_EXPLANATION.")
else:
    # Calculate the reference values from the known synthetic ground-truth mask.
    visible_slices = np.where(xy_cross_sectional_area > 0)[0]
    reference_first = int(visible_slices[0])
    reference_largest = int(np.argmax(xy_cross_sectional_area))
    reference_last = int(visible_slices[-1])

    # Compare each user response with the corresponding reference value.
    checks = {
        "First visible slice": FIRST_VISIBLE_SLICE == reference_first,
        "Largest-area slice": LARGEST_AREA_SLICE == reference_largest,
        "Last visible slice": LAST_VISIBLE_SLICE == reference_last,
    }

    print("Task 1 answer check")
    print("-" * 30)
    for name, is_correct in checks.items():
        print(f"{name}: {'Correct' if is_correct else 'Check again'}")

    print("\nYour explanation:")
    print(YOUR_EXPLANATION)

    if all(checks.values()):
        print("\nExcellent. You correctly identified how the target extends across the XY slices.")
    else:
        print("\nUse the slice explorer and area curve to revise the answers marked 'Check again'.")

# Part 2 — Loading and Understanding a Real CryoET Tomogram

Part 1 used a synthetic sphere to explain the relationship between a 3D object, its 2D cross-sections, and its masks. We will now move from controlled synthetic data to a cropped real CryoET tomogram.

## 2.1 From Tilt Series to Tomogram

A CryoET experiment first records multiple 2D projection images while the frozen sample is viewed from different tilt angles. These projection images are collectively called a **tilt series**.

A reconstruction algorithm combines the tilt series into a 3D tomogram:

$$\text{tilt series}\rightarrow\text{alignment}\rightarrow\text{3D reconstruction}\rightarrow\text{tomogram}.$$

This tutorial will not repeat the alignment and reconstruction process. Instead, we will begin with an already reconstructed tomogram and focus on the next stage:

$$\text{reconstructed tomogram}\rightarrow\text{interactive 3D segmentation}.$$

## 2.2 Real Dataset

The real data used in this tutorial will be selected from the CryoET Data Portal dataset **DS-10003: 70S Ribosome with Chloramphenicol**.

This dataset contains CryoET tomograms of native *Mycoplasma pneumoniae* cells treated with chloramphenicol. It contains 65 tomogram runs together with annotations for cytosolic ribosomes and cellular membranes.

The same underlying EMPIAR-10499 data were used in the CryoSAM paper, making this dataset suitable for connecting the tutorial implementation with the reference method.

- [CryoET Data Portal: DS-10003](https://cryoetdataportal.czscience.com/datasets/10003)
- [EMPIAR-10499](https://www.ebi.ac.uk/empiar/EMPIAR-10499/)
- [CryoSAM paper](https://papers.miccai.org/miccai-2024/paper/0532_paper.pdf)

## 2.3 What Does the Tomogram Contain?

A cellular CryoET tomogram may contain several types of visible structures, including:

- cell membranes;
- ribosomes;
- macromolecular complexes;
- dense cellular material;
- reconstruction artefacts;
- background regions containing noise.

Unlike the synthetic sphere from Part 1, a real ribosome does not have a perfectly smooth boundary or uniform intensity. It may also be surrounded by nearby structures with similar grayscale values.

For this tutorial, we will focus on one compact ribosome-like target. A compact target is suitable for learning prompt propagation because it appears on a limited sequence of neighboring slices and has a more clearly defined spatial extent than a long, connected membrane.

## 2.4 Why Use a Cropped Tomogram?

A complete experimental tomogram can contain hundreds of millions of voxels and may require substantial memory, storage, and download time. Processing a complete dataset would make the notebook slow and unsuitable for an introductory Google Colab tutorial.

Instead, we will use a smaller 3D crop centered around one annotated target:

$$V_{\mathrm{full}}\rightarrow V_{\mathrm{crop}}.$$

The crop remains real CryoET data. Cropping only limits the spatial region processed by the notebook.

A suitable crop must be large enough to include:

- the complete target;
- several slices before the target appears;
- several slices after the target disappears;
- nearby background structures needed to evaluate segmentation errors.

The crop dimensions will be selected using both the target size and the physical voxel spacing. The crop should not be made smaller than the target simply to reduce computation.

## 2.5 Tomogram File Formats

CryoET tomograms are commonly stored in formats such as:

- `.mrc`;
- `.rec`;
- `.map`;
- OME-Zarr.

An MRC file stores the 3D intensity volume together with metadata such as array dimensions and voxel spacing.

After loading, the tomogram is represented in Python as a NumPy array:

$$V\in\mathbb{R}^{D\times H\times W}.$$

The dimension order must always be checked because different tools may use different axis conventions. In this tutorial, the working convention will be:

$$V[z,y,x].$$

This means:

- `volume[z, :, :]` extracts an XY slice;
- `volume[:, y, :]` extracts an XZ slice;
- `volume[:, :, x]` extracts a YZ slice.

## 2.6 Intensity Values in Real CryoET Data

The raw intensity range of a real tomogram may vary between datasets and reconstruction methods. A small number of extreme voxels can also make the full intensity range misleading.

For visualization and model input, we will use percentile normalization:

$$V_{\mathrm{norm}}=\operatorname{clip}\left(\frac{V-p_{\mathrm{low}}}{p_{\mathrm{high}}-p_{\mathrm{low}}},0,1\right),$$

where $p_{\mathrm{low}}$ and $p_{\mathrm{high}}$ are lower and upper intensity percentiles, such as the 1st and 99th percentiles.

Percentile normalization:

- reduces the effect of extreme intensity outliers;
- maps the displayed volume to the range $[0,1]$;
- improves contrast when viewing noisy slices;
- preserves the original spatial arrangement of structures.

Normalization changes the displayed intensity scale but does not change the position or dimensions of the target. The original volume will be retained when physical information is needed.

## 2.7 Annotations and User Prompts Are Different

The selected dataset provides expert annotations for ribosome locations. A particle annotation commonly stores a 3D center coordinate:

$$A=(z_c,y_c,x_c).$$

This annotation helps us select a crop containing a known ribosome. However, a center annotation is not the same as a full segmentation mask.

A center annotation tells us:

> A target is expected near this 3D location.

A 3D segmentation mask tells us:

> These specific voxels belong to the target.

The annotation is also different from the interactive user prompt used later in the tutorial. The dataset annotation is used to prepare and validate the tutorial sample, while the user prompt represents the location selected by the learner during interactive segmentation.

## 2.8 Ground Truth in Synthetic and Real Data

The synthetic volume in Part 1 includes an exact voxel-level ground-truth mask, so its prediction can be evaluated using Dice and IoU.

The real CryoET crop may provide a particle center annotation without an exact voxel-level boundary mask. Therefore, evaluation must be handled differently.

| Data type | Available reference | Suitable evaluation |
|---|---|---|
| Synthetic volume | Exact 3D binary mask | Dice, IoU, volume error, centroid error |
| Real CryoET crop | Expert target coordinate | Center distance, mask continuity, slice span, visual inspection |
| Real data with a voxel mask | Expert 3D segmentation | Dice, IoU, boundary and volume measurements |

We will not create an artificial spherical boundary and present it as an expert segmentation. If an exact voxel-level mask is unavailable, the notebook will clearly distinguish between quantitative reference information and visual quality assessment.

## 2.9 Missing-Wedge Effects

CryoET data are usually collected over a limited range of tilt angles. As a result, part of the 3D Fourier information cannot be measured. This unavailable region is known as the **missing wedge**.

The missing wedge can cause:

- direction-dependent blur;
- elongation along one spatial direction;
- weaker boundaries in some viewing planes;
- inconsistent appearances across XY, XZ, and YZ slices.

This is one reason why a target may be easy to identify in one plane but difficult to identify in another. It also explains why slice-to-slice propagation may fail even when the target is physically continuous.

## 2.10 What We Will Examine in the Real Tomogram

Before placing a prompt or generating a mask, we must first understand the selected real-data crop.

In the next task, we will:

- load the cropped real CryoET tomogram;
- inspect its shape, data type, and voxel spacing;
- normalize its intensity values;
- browse its XY, XZ, and YZ slices;
- locate the annotated target;
- compare the appearance of synthetic and real data;
- select a suitable seed slice for prompt-based segmentation.

No segmentation will be performed yet. The purpose of this part is to understand the real input before asking a model to analyze it.

## Task 2 — Load and Explore a Real CryoET Tomogram Crop

### Task Goal

The goal of this task is to load a real CryoET volume from a public URL, extract a small 3D crop around an annotated ribosome, and inspect the crop from the XY, XZ, and YZ directions.

The data come from **TS_01 / TM-631** in the CryoET Data Portal dataset DS-10003. Instead of downloading the complete 1.55 GB tomogram, the notebook will download only one compressed Zarr chunk of approximately 60 MB and extract a `96 × 96 × 96` crop.

The downloaded chunk is stored temporarily during the notebook session. No tomogram or image file needs to be included in the project repository.

### What You Need to Do

1. Install the small package required to decode the Zarr chunk.
2. Download the real tomogram chunk and ribosome annotation file.
3. Decode the chunk and extract the selected `96³` crop.
4. Inspect the crop shape, data type, voxel spacing, intensity range, and target coordinate.
5. Adjust the display percentiles if the contrast is too weak or too strong.
6. Compare the XY, XZ, and YZ views through the annotated target.
7. Use the interactive viewer to browse neighboring slices.
8. Hide the annotation marker and check whether you can still identify the target.
9. Select an informative plane and seed slice for the next prompt-based segmentation task.
10. Record one difference between the synthetic volume and the real CryoET crop.

### Questions to Consider

- Why is the ribosome less obvious than the synthetic sphere?
- Does the target look equally clear in the XY, XZ, and YZ views?
- How does percentile normalization affect the visibility of weak structures?
- Why should the seed slice pass near the center of the target?
- Why is the expert annotation a point rather than a complete segmentation mask?

### Expected Output

This task will produce:

- download and decoding progress;
- metadata for the real CryoET crop;
- three orthogonal views through the annotated ribosome;
- an interactive slice explorer;
- a selected seed plane and slice;
- a short written observation comparing real and synthetic data.

The green point shown in this task is an expert ribosome annotation. It identifies the approximate target center but does not provide the target boundary or a segmentation mask.

**Data source:** [CryoET Data Portal, DS-10003](https://cryoetdataportal.czscience.com/datasets/10003)  
**Selected run:** [TS_01 / TM-631](https://cryoetdataportal.czscience.com/runs/361?table-tab=Tomograms)

In [ ]:
# Install the codec required to decompress the public Zarr chunk.
%pip install -q numcodecs

In [ ]:
# Import the packages required to download, decode, process, and display the real volume.
import json
import tempfile
import urllib.request
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from numcodecs import Blosc

# Create a temporary cache directory so that downloaded data are not stored in the project repository.
CACHE_DIR = Path(tempfile.gettempdir()) / "interactive_cryoet_tutorial"
CACHE_DIR.mkdir(parents=True, exist_ok=True)

def download_with_progress(url, destination, expected_bytes=None):
    """Download one public file while displaying progress and protecting against incomplete cached files."""
    destination = Path(destination)

    # Remove an incomplete cached file before downloading it again.
    if destination.exists() and expected_bytes is not None and destination.stat().st_size != expected_bytes:
        destination.unlink()

    # Reuse a complete cached file when the cell is run again.
    if destination.exists():
        print(f"Using cached file: {destination.name}")
        return destination

    temporary_path = destination.with_suffix(destination.suffix + ".part")

    def report_progress(block_number, block_size, total_size):
        downloaded = block_number * block_size
        percent = min(100, int(100 * downloaded / total_size)) if total_size > 0 else 0
        print(f"\rDownloading {destination.name}: {percent:3d}%", end="")

    print(f"Connecting to the public data server for {destination.name}...")
    urllib.request.urlretrieve(url, temporary_path, reporthook=report_progress)
    temporary_path.replace(destination)
    print(f"\nDownload complete: {destination.stat().st_size / 1e6:.1f} MB")
    return destination

In [ ]:
# This URL points to one compressed full-resolution Zarr chunk from TS_01 / TM-631.
TOMOGRAM_CHUNK_URL = "https://files.cryoetdataportal.cziscience.com/10003/TS_01/Reconstructions/VoxelSpacing6.802/Tomograms/100/TS_01.zarr/0/0/2/2"

# This URL contains expert-oriented point annotations for the ribosomes in TS_01.
RIBOSOME_ANNOTATION_URL = "https://files.cryoetdataportal.cziscience.com/10003/TS_01/Reconstructions/VoxelSpacing6.802/Annotations/100/chloramphenicol_bound_70s_ribosome-1.0_orientedpoint.ndjson"

# The expected sizes allow the function to detect and replace an interrupted download.
EXPECTED_CHUNK_BYTES = 62183061
EXPECTED_ANNOTATION_BYTES = 111478

chunk_path = download_with_progress(TOMOGRAM_CHUNK_URL, CACHE_DIR / "TS_01_chunk_0_2_2.blosc", EXPECTED_CHUNK_BYTES)
annotation_path = download_with_progress(RIBOSOME_ANNOTATION_URL, CACHE_DIR / "TS_01_ribosomes.ndjson", EXPECTED_ANNOTATION_BYTES)

# Decode the compressed Zarr chunk into one 256 × 256 × 256 NumPy array.
print("Decoding the compressed Zarr chunk...")
compressed_bytes = chunk_path.read_bytes()
decoded_bytes = Blosc().decode(compressed_bytes)
tomogram_chunk = np.frombuffer(decoded_bytes, dtype="<f4").reshape(256, 256, 256)
print(f"Decoded chunk shape: {tomogram_chunk.shape}")

# Read all ribosome annotations and convert their coordinate order from (x, y, z) to (z, y, x).
ribosome_points_zyx = []
with annotation_path.open("r") as file:
    for line in file:
        record = json.loads(line)
        location = record["location"]
        ribosome_points_zyx.append([location["z"], location["y"], location["x"]])

ribosome_points_zyx = np.asarray(ribosome_points_zyx, dtype=np.float32)

# Select the previously reviewed isolated ribosome by finding the annotation nearest to its reference coordinate.
TARGET_REFERENCE_ZYX = np.array([185.55, 644.56, 642.00], dtype=np.float32)
target_annotation_index = int(np.argmin(np.linalg.norm(ribosome_points_zyx - TARGET_REFERENCE_ZYX, axis=1)))
target_global_zyx = ribosome_points_zyx[target_annotation_index]

# The downloaded chunk covers z = 0:256, y = 512:768, and x = 512:768 in the full tomogram.
CHUNK_ORIGIN_ZYX = np.array([0, 512, 512])
target_chunk_zyx = target_global_zyx - CHUNK_ORIGIN_ZYX

# Extract a 96 × 96 × 96 region centered on the selected ribosome.
CROP_SIZE = 96
CROP_HALF_SIZE = CROP_SIZE // 2
crop_center_zyx = np.rint(target_chunk_zyx).astype(int)
crop_start_zyx = crop_center_zyx - CROP_HALF_SIZE
crop_end_zyx = crop_start_zyx + CROP_SIZE

z0, y0, x0 = crop_start_zyx
z1, y1, x1 = crop_end_zyx
real_tomogram_crop = tomogram_chunk[z0:z1, y0:y1, x0:x1].copy()

# Convert the expert annotation into the local coordinate system of the crop.
crop_global_origin_zyx = CHUNK_ORIGIN_ZYX + crop_start_zyx
target_local_zyx = target_global_zyx - crop_global_origin_zyx

# Count the expert ribosome annotations that fall inside this crop for later similar-target experiments.
all_points_local_zyx = ribosome_points_zyx - crop_global_origin_zyx
points_inside_crop = np.all((all_points_local_zyx >= 0) & (all_points_local_zyx < CROP_SIZE), axis=1)
ribosomes_in_crop_zyx = all_points_local_zyx[points_inside_crop]

VOXEL_SPACING_ANGSTROM = 6.802

print("Real CryoET crop created successfully.")
print(f"Crop shape: {real_tomogram_crop.shape}")
print(f"Data type: {real_tomogram_crop.dtype}")
print(f"Voxel spacing: {VOXEL_SPACING_ANGSTROM} Å")
print(f"Raw intensity range: [{real_tomogram_crop.min():.4f}, {real_tomogram_crop.max():.4f}]")
print(f"Selected annotation index: {target_annotation_index}")
print(f"Target coordinate in full tomogram, zyx: {np.round(target_global_zyx, 2)}")
print(f"Target coordinate in crop, zyx: {np.round(target_local_zyx, 2)}")
print(f"Annotated ribosomes inside crop: {len(ribosomes_in_crop_zyx)}")

In [ ]:
def percentile_normalize(volume, low_percentile=1.0, high_percentile=99.0):
    """Normalize a volume to [0, 1] using two user-selected intensity percentiles."""
    low_value, high_value = np.percentile(volume, [low_percentile, high_percentile])

    if high_value <= low_value:
        raise ValueError("HIGH_PERCENTILE must be greater than LOW_PERCENTILE.")

    normalized = np.clip((volume - low_value) / (high_value - low_value), 0, 1)
    return normalized, low_value, high_value

# Change these values and rerun the cell if the real-data contrast is too weak or too strong.
LOW_PERCENTILE = 1.0  # WriteYourCodeHere: Try a value such as 0.5, 1.0, or 2.0.
HIGH_PERCENTILE = 99.0  # WriteYourCodeHere: Try a value such as 98.0, 99.0, or 99.5.

# Invert the display only if you find dark structures easier to inspect on a light background.
DISPLAY_INVERTED = False  # WriteYourCodeHere: Change this to True and compare the result.

real_tomogram_normalized, display_low, display_high = percentile_normalize(real_tomogram_crop, LOW_PERCENTILE, HIGH_PERCENTILE)
real_tomogram_display = 1.0 - real_tomogram_normalized if DISPLAY_INVERTED else real_tomogram_normalized

print(f"Display percentiles: {LOW_PERCENTILE}% to {HIGH_PERCENTILE}%")
print(f"Raw values mapped to [0, 1]: [{display_low:.4f}, {display_high:.4f}]")
print(f"Display inverted: {DISPLAY_INVERTED}")

In [ ]:
# Round the expert coordinate only when selecting discrete array slices.
target_z, target_y, target_x = np.rint(target_local_zyx).astype(int)

# Extract the three orthogonal planes that intersect at the annotated target.
xy_slice = real_tomogram_display[target_z, :, :]
xz_slice = real_tomogram_display[:, target_y, :]
yz_slice = real_tomogram_display[:, :, target_x]

fig, axes = plt.subplots(1, 3, figsize=(14, 4))

axes[0].imshow(xy_slice, cmap="gray", origin="lower")
axes[0].scatter(target_x, target_y, c="lime", s=45, edgecolors="black", linewidths=0.6)
axes[0].set_title(f"XY view at z = {target_z}")
axes[0].set_xlabel("x")
axes[0].set_ylabel("y")

axes[1].imshow(xz_slice, cmap="gray", origin="lower")
axes[1].scatter(target_x, target_z, c="lime", s=45, edgecolors="black", linewidths=0.6)
axes[1].set_title(f"XZ view at y = {target_y}")
axes[1].set_xlabel("x")
axes[1].set_ylabel("z")

axes[2].imshow(yz_slice, cmap="gray", origin="lower")
axes[2].scatter(target_y, target_z, c="lime", s=45, edgecolors="black", linewidths=0.6)
axes[2].set_title(f"YZ view at x = {target_x}")
axes[2].set_xlabel("y")
axes[2].set_ylabel("z")

plt.tight_layout()
plt.show()

In [ ]:
def extract_real_slice(volume, plane, slice_index):
    """Extract one slice from the real CryoET crop using the selected plane."""
    if plane == "XY":
        return volume[slice_index, :, :]
    if plane == "XZ":
        return volume[:, slice_index, :]
    if plane == "YZ":
        return volume[:, :, slice_index]
    raise ValueError("Plane must be 'XY', 'XZ', or 'YZ'.")

def show_real_tomogram_slice(plane="XY", slice_index=48, show_annotation=True):
    """Display one real CryoET slice and optionally show the expert target annotation."""
    image_slice = extract_real_slice(real_tomogram_display, plane, slice_index)

    fig, ax = plt.subplots(figsize=(6, 6))
    ax.imshow(image_slice, cmap="gray", origin="lower")

    # Show the target only when the selected slice passes through its annotated 3D coordinate.
    if plane == "XY" and show_annotation and abs(slice_index - target_z) <= 1:
        ax.scatter(target_x, target_y, c="lime", s=55, edgecolors="black", linewidths=0.7)
    elif plane == "XZ" and show_annotation and abs(slice_index - target_y) <= 1:
        ax.scatter(target_x, target_z, c="lime", s=55, edgecolors="black", linewidths=0.7)
    elif plane == "YZ" and show_annotation and abs(slice_index - target_x) <= 1:
        ax.scatter(target_y, target_z, c="lime", s=55, edgecolors="black", linewidths=0.7)

    ax.set_title(f"Real CryoET crop — {plane} slice {slice_index}")
    ax.set_xlabel("Horizontal image coordinate")
    ax.set_ylabel("Vertical image coordinate")
    plt.tight_layout()
    plt.show()

# Choose the initial state used when the interactive viewer first opens.
INITIAL_REAL_PLANE = "XY"  # WriteYourCodeHere: Change this to "XZ" or "YZ" if desired.
INITIAL_REAL_SLICE = target_z  # WriteYourCodeHere: Replace this with another slice index to begin elsewhere.

try:
    import ipywidgets as widgets
    from IPython.display import display

    plane_control = widgets.Dropdown(options=["XY", "XZ", "YZ"], value=INITIAL_REAL_PLANE, description="Plane:")
    slice_control = widgets.IntSlider(value=int(INITIAL_REAL_SLICE), min=0, max=CROP_SIZE - 1, step=1, description="Slice:", continuous_update=False)
    annotation_control = widgets.Checkbox(value=True, description="Show annotation")
    interactive_viewer = widgets.interactive_output(show_real_tomogram_slice, {"plane": plane_control, "slice_index": slice_control, "show_annotation": annotation_control})

    display(widgets.HBox([plane_control, slice_control, annotation_control]), interactive_viewer)
except ImportError:
    print("ipywidgets is not available. Change INITIAL_REAL_PLANE and INITIAL_REAL_SLICE, then call show_real_tomogram_slice manually.")
    show_real_tomogram_slice(INITIAL_REAL_PLANE, int(INITIAL_REAL_SLICE), True)

# Part 3 — From a User Prompt to an Initial 2D Mask

A CryoET tomogram may contain many visually similar structures. Even after selecting a slice, a segmentation model cannot automatically know which object the user wants to isolate. We therefore provide a **visual prompt** that identifies the target.

This follows the promptable segmentation idea introduced by the Segment Anything Model (SAM) and adapted to CryoET analysis by CryoSAM.

## 3.1 What Is a Visual Prompt?

A visual prompt is a spatial instruction placed directly on an image. It is different from the slice index: the slice index selects an image, while the prompt tells the model **which structure inside that image** should be segmented.

Three common prompt types are:

- **Positive point:** a point placed inside the target structure.
- **Negative point:** a point placed on the background or an unwanted neighboring structure.
- **Bounding box:** a rectangle that approximately encloses the target.

A point prompt can be represented as:

$$p_i=(x_i,y_i,l_i)$$

Here, $(x_i,y_i)$ is the pixel coordinate and $l_i$ is the prompt label:

$$l_i=\begin{cases}1,&\text{positive point}\\0,&\text{negative point}\end{cases}$$

A complete prompt set on the selected seed slice $I_s$ is written as:

$$P_s=\{p_1,p_2,\ldots,p_n\}$$

## 3.2 Positive and Negative Points

A positive point tells the model:

> “The target should include the structure at this location.”

A negative point tells the model:

> “This location should not be included in the target.”

For an isolated object, one positive point near its center may be sufficient. If the predicted mask includes a neighboring object, a negative point can be placed on that unwanted region. If part of the target is missing, another positive point can be added to the missing region.

During interaction step $t$, a new point $p_t$ is added to the existing prompt set:

$$P_s^{(t)}=P_s^{(t-1)}\cup\{p_t\}$$

The model then updates its prediction using all available prompts.

## 3.3 How Prompt-Based Segmentation Works

A promptable segmentation model receives both the selected image and the user prompts:

$$S_s=f(I_s,P_s)$$

Here:

- $I_s$ is the selected 2D tomogram slice.
- $P_s$ is the set of positive and negative prompts.
- $f$ is the segmentation model.
- $S_s(x,y)$ is the predicted foreground score at pixel $(x,y)$.

In a SAM-style model, the **image encoder** extracts visual features from the slice, the **prompt encoder** converts points or boxes into prompt features, and the **mask decoder** combines both types of features to predict the target region.

The model output is usually a score or probability map rather than an immediate binary mask. A threshold $\tau$ converts this score map into the initial 2D mask:

$$M_s(x,y)=\begin{cases}1,&S_s(x,y)\geq\tau\\0,&S_s(x,y)<\tau\end{cases}$$

Pixels labeled 1 belong to the predicted target, while pixels labeled 0 are treated as background.

## 3.4 Why Is CryoET Segmentation Difficult?

Prompt-based segmentation is more difficult in CryoET than in ordinary photographs because tomogram slices usually contain:

- low signal-to-noise ratio;
- weak or incomplete object boundaries;
- reconstruction artifacts;
- nearby structures with similar intensity;
- anisotropic information caused by the missing wedge.

A point prompt identifies the intended object, but it does not directly describe the object's boundary. The segmentation method must infer that boundary from the local image appearance. Prompt placement and seed-slice selection can therefore strongly affect the result.

## 3.5 Choosing a Good Seed Slice

The first prompted slice is called the **seed slice**. A useful seed slice should ideally:

- contain a visible cross-section of the target;
- show a relatively clear target boundary;
- separate the target from nearby structures;
- pass near the center of the 3D object.

A central slice is usually preferable because the target occupies more pixels and provides more structural information. A slice near the top or bottom of the object may contain only a small and ambiguous cross-section.

## 3.6 What Is the Initial 2D Mask?

The initial 2D mask $M_s$ is the segmentation predicted on the seed slice after the user provides a prompt. It is called “initial” because it covers only one slice and may still require correction.

| Mask | Meaning |
|---|---|
| Ground-truth mask | A manually verified reference annotation |
| Initial 2D mask | The first prediction on the prompted seed slice |
| Propagated 3D mask | The final volume assembled from predictions across multiple slices |

The initial mask is not assumed to be correct. The user should inspect whether it includes the intended target, excludes nearby structures, follows the visible target boundary, and avoids obvious holes or disconnected regions.

If necessary, additional positive or negative prompts can be added to refine the prediction.

## 3.7 From One 2D Mask to a 3D Structure

A point only identifies a location, while a mask describes the target's approximate size, shape, and boundary on the seed slice. The initial mask therefore provides richer guidance for the neighboring slices.

In the next part, the current mask will guide segmentation on the next and previous slices:

$$M_{s+1}=g(I_{s+1},M_s)$$

$$M_{s-1}=g(I_{s-1},M_s)$$

This process continues in both directions until the target disappears or the prediction becomes unreliable. The resulting 2D masks can then be stacked to form a 3D mask.

## What You Will Do in Task 3

You will:

1. select a seed plane and seed slice;
2. click inside the ribosome to create a positive prompt;
3. generate an initial 2D segmentation;
4. optionally add positive or negative corrective prompts;
5. inspect the prompt, score map, and binary mask;
6. save the accepted mask for later bidirectional propagation.

## Task 3 — Create an Initial 2D Mask with Visual Prompts

### Goal

In this task, you will interactively identify one ribosome on the selected seed slice and generate its initial 2D mask.

You will first place prompts and then run a prompt-guided segmentation method. Keeping these two stages separate allows you to see how changing the user input changes the prediction.

### Educational Segmentation Method

Running the complete SAM or CryoSAM model requires additional model weights and substantially more memory. To keep this notebook lightweight and interactive, we use a **Random Walker** segmentation method as an educational prompt-based baseline.

Random Walker treats the image as a graph:

- neighboring pixels are connected;
- pixels with similar intensities have stronger connections;
- positive prompts act as foreground seeds;
- negative prompts and image borders act as background seeds.

The method estimates a foreground probability for every pixel. We then apply a threshold $\tau$ to obtain the initial mask:

$$M_s(x,y)=\mathbf{1}[S_s(x,y)\geq\tau]$$

Although this is not the neural network used by SAM, it preserves the same interaction pattern:

$$\text{image}+\text{user prompts}\rightarrow\text{score map}\rightarrow\text{binary mask}$$

The segmentation method can therefore be replaced by SAM or another promptable model without changing the rest of the tutorial pipeline.

### Your Tasks

1. Choose the seed plane and seed slice.
2. Use the coordinate controls to place a positive prompt inside the ribosome.
3. Run the segmentation cell to generate the initial mask.
4. Inspect whether the mask follows the intended target.
5. Add another positive point if part of the target is missing.
6. Add a negative point if the mask includes a nearby structure.
7. Adjust the mask threshold and compare the results.

### Expected Output

The final visualization will show:

- the selected slice and user prompts;
- the predicted foreground score map;
- the initial binary mask overlaid on the tomogram slice.

A useful initial mask should cover one ribosome without spreading into neighboring structures. It does not need to be perfect, but it should provide a reliable starting point for bidirectional propagation.

In [ ]:
%pip install -q scikit-image

import ipywidgets as widgets
from IPython.display import display, clear_output

SEED_PLANE = "XY"  # WriteYourCodeHere: choose "XY", "XZ", or "YZ".
SEED_SLICE = target_z  # WriteYourCodeHere: choose the seed slice identified in Part 2.

# Extract the seed slice from the normalized real CryoET crop created in Part 2.
seed_image = extract_real_slice(real_tomogram_display, SEED_PLANE, SEED_SLICE)
prompt_points, prompt_labels = [], []

# Start the controls near the expert annotation while allowing the user to move the prompt.
if SEED_PLANE == "XY":
    initial_x, initial_y = target_x, target_y
elif SEED_PLANE == "XZ":
    initial_x, initial_y = target_x, target_z
else:
    initial_x, initial_y = target_y, target_z

prompt_x = widgets.IntSlider(value=int(initial_x), min=0, max=seed_image.shape[1] - 1, description="x:")
prompt_y = widgets.IntSlider(value=int(initial_y), min=0, max=seed_image.shape[0] - 1, description="y:")
prompt_type = widgets.Dropdown(options=[("Positive (+)", 1), ("Negative (−)", 0)], value=1, description="Type:")
add_button = widgets.Button(description="Add prompt", button_style="success")
undo_button = widgets.Button(description="Undo")
clear_button = widgets.Button(description="Clear")
prompt_output = widgets.Output()

def draw_prompt_editor(change=None):
    with prompt_output:
        clear_output(wait=True)
        fig, ax = plt.subplots(figsize=(5, 5))
        ax.imshow(seed_image, cmap="gray", origin="lower")
        ax.scatter(prompt_x.value, prompt_y.value, s=100, marker="+", c="yellow", linewidths=2)
        for (x, y), label_value in zip(prompt_points, prompt_labels):
            ax.scatter(x, y, s=70, c="lime" if label_value == 1 else "red", edgecolors="black")
        ax.set_title(f"{SEED_PLANE} slice {SEED_SLICE} | {len(prompt_points)} saved prompt(s)")
        ax.set_xlim(-0.5, seed_image.shape[1] - 0.5)
        ax.set_ylim(-0.5, seed_image.shape[0] - 0.5)
        ax.set_xlabel("x")
        ax.set_ylabel("y")
        plt.tight_layout()
        plt.show()

def add_prompt(_):
    prompt_points.append((prompt_x.value, prompt_y.value))
    prompt_labels.append(prompt_type.value)
    draw_prompt_editor()

def undo_prompt(_):
    if prompt_points:
        prompt_points.pop()
        prompt_labels.pop()
    draw_prompt_editor()

def clear_prompts(_):
    prompt_points.clear()
    prompt_labels.clear()
    draw_prompt_editor()

add_button.on_click(add_prompt)
undo_button.on_click(undo_prompt)
clear_button.on_click(clear_prompts)
prompt_x.observe(draw_prompt_editor, names="value")
prompt_y.observe(draw_prompt_editor, names="value")
prompt_type.observe(draw_prompt_editor, names="value")

display(widgets.VBox([widgets.HBox([prompt_type, add_button, undo_button, clear_button]), prompt_x, prompt_y]), prompt_output)
draw_prompt_editor()

In [ ]:
from skimage.measure import label
from skimage.segmentation import random_walker

SMOOTHING_SIGMA = 1.0  # WriteYourCodeHere: increase this value if the image is too noisy.
RANDOM_WALKER_BETA = 130.0  # WriteYourCodeHere: increase this value to make intensity boundaries more important.
MASK_THRESHOLD = 0.50  # WriteYourCodeHere: decrease it for a larger mask or increase it for a smaller mask.
PROMPT_RADIUS = 2  # WriteYourCodeHere: choose the radius of each prompt seed.

if 1 not in prompt_labels:
    raise ValueError("Add at least one positive prompt in the previous cell before running segmentation.")
if not 0 < MASK_THRESHOLD < 1:
    raise ValueError("MASK_THRESHOLD must be between 0 and 1.")

# Smooth the noisy CryoET slice before applying prompt-guided segmentation.
filtered_seed_image = gaussian_filter(seed_image, sigma=SMOOTHING_SIGMA)

# Create the Random Walker seed map: 0 = unknown, 1 = background, and 2 = foreground.
seed_labels = np.zeros(seed_image.shape, dtype=np.uint8)
seed_labels[:2, :] = 1
seed_labels[-2:, :] = 1
seed_labels[:, :2] = 1
seed_labels[:, -2:] = 1
yy, xx = np.ogrid[:seed_image.shape[0], :seed_image.shape[1]]

# Convert each point prompt into a small foreground or background seed region.
for (x, y), prompt_label in zip(prompt_points, prompt_labels):
    prompt_disk = (xx - x) ** 2 + (yy - y) ** 2 <= PROMPT_RADIUS ** 2
    seed_labels[prompt_disk] = 2 if prompt_label == 1 else 1

# Estimate the foreground probability of every pixel.
probability_maps = random_walker(filtered_seed_image, seed_labels, beta=RANDOM_WALKER_BETA, mode="bf", return_full_prob=True)
initial_score_map = probability_maps[1]

# Apply the selected threshold and retain only components connected to positive prompts.
candidate_mask = initial_score_map >= MASK_THRESHOLD
component_map = label(candidate_mask)
positive_component_ids = {component_map[y, x] for (x, y), prompt_label in zip(prompt_points, prompt_labels) if prompt_label == 1 and component_map[y, x] != 0}
initial_2d_mask = np.isin(component_map, list(positive_component_ids))

# Compare the prompts, foreground scores, and resulting initial mask.
fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(seed_image, cmap="gray", origin="lower")
for (x, y), prompt_label in zip(prompt_points, prompt_labels):
    axes[0].scatter(x, y, s=70, c="lime" if prompt_label == 1 else "red", edgecolors="black")
axes[0].set_title("Seed Slice and Prompts")
score_view = axes[1].imshow(initial_score_map, cmap="viridis", vmin=0, vmax=1, origin="lower")
axes[1].set_title("Foreground Score Map")
fig.colorbar(score_view, ax=axes[1], fraction=0.046, pad=0.04)
axes[2].imshow(seed_image, cmap="gray", origin="lower")
axes[2].imshow(np.ma.masked_where(~initial_2d_mask, initial_2d_mask), cmap="autumn", alpha=0.45, origin="lower")
axes[2].contour(initial_2d_mask, levels=[0.5], colors="yellow", linewidths=1, origin="lower")
axes[2].set_title(f"Initial 2D Mask | {initial_2d_mask.sum()} pixels")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Positive prompts: {prompt_labels.count(1)}")
print(f"Negative prompts: {prompt_labels.count(0)}")
print(f"Foreground pixels: {initial_2d_mask.sum()}")

# Part 4 — Automatic Prompt Generation from the Current Mask

After Task 3, we have an accepted mask $M_s$ on the seed slice $I_s$. To continue segmentation without requesting another manual prompt, the system must convert this mask into an automatic prompt for a neighboring slice.

This process can be written as:

$$M_s\xrightarrow{h}P_{s+d}\xrightarrow{f(I_{s+d},\,P_{s+d})}M_{s+d}$$

Here, $d\in\{-1,+1\}$ specifies the propagation direction, $h$ converts the current mask into an automatic prompt, and $f$ segments the neighboring slice using that prompt.

This is a lightweight educational version of the self-prompting idea used in CryoSAM.

## 4.1 Why Can a Mask Guide a Neighboring Slice?

Neighboring tomogram slices are spatially aligned. A pixel at coordinate $(x,y)$ on slice $s$ corresponds approximately to the same physical location on slices $s-1$ and $s+1$.

If a target occupies region $M_s$ on the current slice, the target will often appear near the same coordinates on an adjacent slice. However, its cross-sectional shape and size may change.

Therefore, we should not directly copy the current mask:

$$M_{s+d}\neq M_s$$

Instead, the current mask provides spatial guidance, while the neighboring image determines the new boundary:

$$M_{s+d}=f(I_{s+d},P_{s+d})$$

## 4.2 Extracting a Reliable Foreground Prompt

Pixels near the boundary of a predicted mask are usually less reliable than pixels near its center. We therefore erode the current mask to create a conservative foreground core:

$$C_s=\operatorname{Erode}(M_s,r_c)$$

Here, $r_c$ is the erosion radius. The resulting core $C_s$ contains pixels that are likely to remain inside the target on the neighboring slice.

If erosion removes a very small mask completely, the system can instead select the most interior point. Let $D_s(x,y)$ be the distance from an internal mask pixel to the nearest boundary. The automatic positive point is:

$$p_s^+=\underset{(x,y):M_s(x,y)=1}{\operatorname{argmax}}D_s(x,y)$$

This point is more reliable than the mask centroid because a centroid may fall outside an irregular or disconnected mask.

## 4.3 Extracting a Background Prompt

The system also needs background information to prevent the next mask from spreading into nearby structures.

A background ring can be generated outside the current mask:

$$B_s=\operatorname{Dilate}(M_s,r_o)\setminus\operatorname{Dilate}(M_s,r_g),\qquad r_o>r_g$$

Here:

- $r_g$ creates an uncertain gap around the target boundary;
- $r_o$ defines the outer radius of the background ring;
- pixels in $B_s$ are used as background seeds.

The gap is important because the target may become slightly larger on the neighboring slice. Marking pixels immediately outside the current boundary as background could incorrectly prevent this natural growth.

## 4.4 Transferring the Prompt

The foreground core and background ring are transferred to the same image coordinates on the neighboring slice:

$$P_{s+d}=\{C_s,B_s\}$$

The automatic prompt contains more information than the original user point:

| Prompt component | Meaning |
|---|---|
| Foreground core $C_s$ | Region that should remain inside the target |
| Background ring $B_s$ | Region that should remain outside the target |
| Neighboring image $I_{s+d}$ | Image evidence used to determine the new boundary |

The prompt is transferred, but the previous mask is not treated as the final answer for the next slice.

## 4.5 Predicting the Neighboring Mask

The prompt-guided segmentation method produces a new score map:

$$S_{s+d}=f(I_{s+d},C_s,B_s)$$

A threshold $\tau$ converts the score map into the next binary mask:

$$M_{s+d}(x,y)=\mathbf{1}[S_{s+d}(x,y)\geq\tau]$$

Because the score map is calculated from the neighboring image, the new mask can expand, shrink, or change shape relative to $M_s$.

The newly predicted mask then becomes the current mask for the next propagation step:

$$M_s\rightarrow P_{s+d}\rightarrow M_{s+d}\rightarrow P_{s+2d}\rightarrow M_{s+2d}$$

Part 4 will perform only the first transition. Part 5 will repeat it in both directions.

## 4.6 Sources of Propagation Error

Automatic prompting assumes that the target changes gradually between neighboring slices. It may fail when:

- the current mask already includes an incorrect structure;
- the target moves rapidly between slices;
- the target becomes very small or disappears;
- the foreground core becomes empty;
- the background ring overlaps the true target;
- noise or reconstruction artifacts create a false boundary.

For this reason, later propagation must include stopping criteria rather than continuing through every slice.

## What You Will Do in Task 4

You will:

1. use `initial_2d_mask` as the current mask;
2. generate a conservative foreground core;
3. generate a background ring with an uncertain gap;
4. transfer these automatic prompts to one neighboring slice;
5. segment that slice to obtain a one-step prediction;
6. compare the current mask with the neighboring mask;
7. adjust the prompt-generation parameters and observe how they affect the result.

## Task 4 — Generate and Transfer an Automatic Prompt

### Goal

In this task, you will convert the accepted initial mask into an automatic prompt for one neighboring slice.

Unlike Task 3, you will not manually select another point. The system will automatically extract:

- a reliable foreground core from inside the current mask;
- a background ring outside the current mask;
- the corresponding prompt coordinates on the neighboring slice.

The neighboring image will then be segmented using these automatically generated prompts.

### Your Tasks

1. Choose a propagation direction: `+1` or `-1`.
2. Adjust the foreground erosion radius.
3. Adjust the background gap and ring width.
4. Inspect where the automatic prompts are placed.
5. Generate a one-step segmentation on the neighboring slice.
6. Compare the current mask and the neighboring mask.

### Parameters to Explore

- `CORE_EROSION`: controls how conservatively the foreground prompt is extracted.
- `BACKGROUND_GAP`: leaves uncertain space around the target boundary.
- `BACKGROUND_WIDTH`: controls the width of the background prompt.
- `NEXT_MASK_THRESHOLD`: controls the size of the predicted neighboring mask.

### Expected Output

The first code cell will visualize the current mask, foreground core, background ring, and transferred prompt. The second code cell will show the neighboring score map, the predicted neighboring mask, and the change between the two slices.

In [ ]:
from scipy.ndimage import binary_dilation, binary_erosion, distance_transform_edt

PROPAGATION_DIRECTION = 1  # WriteYourCodeHere: use 1 for the next slice or -1 for the previous slice.
CORE_EROSION = 2  # WriteYourCodeHere: increase this value to create a more conservative foreground core.
BACKGROUND_GAP = 3  # WriteYourCodeHere: leave uncertain space around the current boundary.
BACKGROUND_WIDTH = 3  # WriteYourCodeHere: control the width of the background ring.

if PROPAGATION_DIRECTION not in (-1, 1):
    raise ValueError("PROPAGATION_DIRECTION must be 1 or -1.")
if CORE_EROSION < 0 or BACKGROUND_GAP < 0 or BACKGROUND_WIDTH < 1:
    raise ValueError("Prompt-generation parameters must be non-negative and BACKGROUND_WIDTH must be at least 1.")

current_mask = initial_2d_mask.astype(bool)
if not current_mask.any():
    raise ValueError("The initial 2D mask is empty. Return to Task 3 and adjust the prompts or threshold.")

plane_size = {"XY": real_tomogram_display.shape[0], "XZ": real_tomogram_display.shape[1], "YZ": real_tomogram_display.shape[2]}[SEED_PLANE]
next_slice_index = SEED_SLICE + PROPAGATION_DIRECTION
if not 0 <= next_slice_index < plane_size:
    raise ValueError("The requested neighboring slice is outside the tomogram crop.")

# Erode the current mask to obtain a reliable foreground region.
foreground_core = binary_erosion(current_mask, iterations=CORE_EROSION) if CORE_EROSION > 0 else current_mask.copy()

# Find the most interior target pixel and use it if erosion removes the complete mask.
distance_inside = distance_transform_edt(current_mask)
automatic_positive_y, automatic_positive_x = np.unravel_index(np.argmax(distance_inside), distance_inside.shape)
if not foreground_core.any():
    foreground_core[automatic_positive_y, automatic_positive_x] = True

# Create a background ring while preserving an uncertain gap around the target.
background_inner = binary_dilation(current_mask, iterations=BACKGROUND_GAP) if BACKGROUND_GAP > 0 else current_mask.copy()
background_outer = binary_dilation(current_mask, iterations=BACKGROUND_GAP + BACKGROUND_WIDTH)
background_ring = background_outer & ~background_inner

# Transfer the mask-derived prompts to the same coordinates on the neighboring slice.
next_image = extract_real_slice(real_tomogram_display, SEED_PLANE, next_slice_index)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
axes[0].imshow(seed_image, cmap="gray", origin="lower")
axes[0].imshow(np.ma.masked_where(~current_mask, current_mask), cmap="autumn", alpha=0.45, origin="lower")
axes[0].set_title(f"Current Mask | Slice {SEED_SLICE}")
axes[1].imshow(seed_image, cmap="gray", origin="lower")
axes[1].imshow(np.ma.masked_where(~foreground_core, foreground_core), cmap="Greens", alpha=0.75, origin="lower")
axes[1].imshow(np.ma.masked_where(~background_ring, background_ring), cmap="Reds", alpha=0.55, origin="lower")
axes[1].scatter(automatic_positive_x, automatic_positive_y, c="lime", s=45, edgecolors="black")
axes[1].set_title("Automatic Prompt")
axes[2].imshow(next_image, cmap="gray", origin="lower")
axes[2].imshow(np.ma.masked_where(~foreground_core, foreground_core), cmap="Greens", alpha=0.75, origin="lower")
axes[2].imshow(np.ma.masked_where(~background_ring, background_ring), cmap="Reds", alpha=0.55, origin="lower")
axes[2].set_title(f"Transferred Prompt | Slice {next_slice_index}")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Propagation: slice {SEED_SLICE} → slice {next_slice_index}")
print(f"Foreground prompt pixels: {foreground_core.sum()}")
print(f"Background prompt pixels: {background_ring.sum()}")

In [ ]:
NEXT_MASK_THRESHOLD = MASK_THRESHOLD  # WriteYourCodeHere: decrease it for a larger neighboring mask or increase it for a smaller mask.

if not 0 < NEXT_MASK_THRESHOLD < 1:
    raise ValueError("NEXT_MASK_THRESHOLD must be between 0 and 1.")

# Build the automatic seed map: 0 = unknown, 1 = background, and 2 = foreground.
automatic_seed_labels = np.zeros(next_image.shape, dtype=np.uint8)
automatic_seed_labels[:2, :] = 1
automatic_seed_labels[-2:, :] = 1
automatic_seed_labels[:, :2] = 1
automatic_seed_labels[:, -2:] = 1
automatic_seed_labels[background_ring] = 1
automatic_seed_labels[foreground_core] = 2

# Segment the neighboring slice using the automatically transferred prompts.
filtered_next_image = gaussian_filter(next_image, sigma=SMOOTHING_SIGMA)
next_probability_maps = random_walker(filtered_next_image, automatic_seed_labels, beta=RANDOM_WALKER_BETA, mode="bf", return_full_prob=True)
next_score_map = next_probability_maps[1]
next_candidate_mask = next_score_map >= NEXT_MASK_THRESHOLD

# Retain only predicted components connected to the transferred foreground core.
next_component_map = label(next_candidate_mask)
connected_component_ids = np.unique(next_component_map[foreground_core])
connected_component_ids = connected_component_ids[connected_component_ids != 0]
next_2d_mask = np.isin(next_component_map, connected_component_ids)

current_area = int(current_mask.sum())
next_area = int(next_2d_mask.sum())
intersection = int(np.logical_and(current_mask, next_2d_mask).sum())
union = int(np.logical_or(current_mask, next_2d_mask).sum())
adjacent_iou = intersection / union if union > 0 else 0.0
area_ratio = next_area / current_area if current_area > 0 else 0.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))
score_view = axes[0].imshow(next_score_map, cmap="viridis", vmin=0, vmax=1, origin="lower")
axes[0].set_title(f"Neighbor Score Map | Slice {next_slice_index}")
fig.colorbar(score_view, ax=axes[0], fraction=0.046, pad=0.04)
axes[1].imshow(next_image, cmap="gray", origin="lower")
axes[1].imshow(np.ma.masked_where(~next_2d_mask, next_2d_mask), cmap="autumn", alpha=0.45, origin="lower")
axes[1].contour(next_2d_mask, levels=[0.5], colors="yellow", linewidths=1, origin="lower")
axes[1].set_title(f"Neighbor Mask | {next_area} pixels")
axes[2].imshow(next_image, cmap="gray", origin="lower")
axes[2].contour(current_mask, levels=[0.5], colors="yellow", linewidths=1, origin="lower")
axes[2].contour(next_2d_mask, levels=[0.5], colors="cyan", linewidths=1, origin="lower")
axes[2].set_title("Yellow: Current | Cyan: Next")
for ax in axes:
    ax.axis("off")
plt.tight_layout()
plt.show()

print(f"Current mask area: {current_area} pixels")
print(f"Neighbor mask area: {next_area} pixels")
print(f"Area ratio: {area_ratio:.3f}")
print(f"Adjacent-mask IoU: {adjacent_iou:.3f}")

# Part 5 — Bidirectional Mask Propagation

Part 4 demonstrated one propagation step:

$$M_s\rightarrow P_{s+d}\rightarrow M_{s+d}$$

where the current mask generates an automatic prompt and the neighboring image determines a new mask.

In Part 5, this operation is repeated in both directions from the seed slice:

$$s\rightarrow s+1\rightarrow s+2\rightarrow\cdots$$

and

$$s\rightarrow s-1\rightarrow s-2\rightarrow\cdots$$

This process is called **bidirectional propagation**.

## 5.1 Why Propagate in Both Directions?

A 3D target usually extends across multiple slices. The manually selected seed slice is often located near the center of the target, but it does not necessarily correspond to its first or last appearance.

Forward propagation follows the target toward increasing slice indices:

$$M_s\rightarrow M_{s+1}\rightarrow M_{s+2}\rightarrow\cdots$$

Backward propagation follows it toward decreasing slice indices:

$$M_s\rightarrow M_{s-1}\rightarrow M_{s-2}\rightarrow\cdots$$

Combining the two directions allows the system to recover the complete slice span of the target.

## 5.2 Iterative Self-Prompting

At every propagation step, the newly accepted mask becomes the source of the next automatic prompt.

For direction $d\in\{-1,+1\}$, the iterative process is:

$$P_{k+d}=h(M_k)$$

$$S_{k+d}=f(I_{k+d},P_{k+d})$$

$$\hat{M}_{k+d}=\mathbf{1}[S_{k+d}\geq\tau]$$

Here:

- $M_k$ is the current accepted mask;
- $h$ extracts the foreground core and background ring;
- $I_{k+d}$ is the neighboring slice;
- $S_{k+d}$ is its foreground score map;
- $\hat{M}_{k+d}$ is the candidate mask.

If the candidate passes the reliability checks, it is accepted:

$$M_{k+d}=\hat{M}_{k+d}$$

The process then continues from this newly accepted mask.

Therefore, propagation uses a moving reference:

$$M_s\rightarrow M_{s+1}\rightarrow M_{s+2}$$

rather than repeatedly generating every mask directly from $M_s$.

## 5.3 Why Are Stopping Criteria Necessary?

The target does not occupy every slice in the tomogram. As propagation approaches the end of the target, its cross-section usually becomes smaller and eventually disappears.

Propagation can also fail because of noise, nearby structures, or an incorrect mask from an earlier step. If the system continues without checking the result, one error may be transferred into all subsequent slices.

Part 5 therefore evaluates each candidate mask before accepting it.

### Empty-mask criterion

Propagation stops if no foreground region is predicted:

$$|\hat{M}_{k+d}|=0$$

This may indicate that the target has disappeared or that segmentation has failed.

### Area-ratio criterion

The area ratio between consecutive masks is:

$$R_A=\frac{|\hat{M}_{k+d}|}{|M_k|}$$

A moderate change is expected because the target cross-section may naturally expand or shrink. However, an extremely large or small ratio may indicate propagation failure.

The candidate is accepted only when:

$$R_{\min}\leq R_A\leq R_{\max}$$

### Adjacent-mask overlap

The spatial overlap between consecutive masks can be measured using intersection over union:

$$\operatorname{IoU}(M_k,\hat{M}_{k+d})
=
\frac{|M_k\cap\hat{M}_{k+d}|}
{|M_k\cup\hat{M}_{k+d}|}
$$

A very low IoU suggests that the predicted mask has moved away from the previous target.

Adjacent-mask IoU is a continuity measurement, not segmentation accuracy, because no ground-truth mask is available for the neighboring slice.

### Centroid-displacement criterion

For a mask $M$, its centroid is:

$$\mathbf{c}(M)=(\bar{x},\bar{y})$$

The displacement between two neighboring masks is:

$$D_C=
\left\|
\mathbf{c}(M_k)-\mathbf{c}(\hat{M}_{k+d})
\right\|_2
$$

A large displacement may indicate that propagation has switched to another nearby structure.

## 5.4 Accepting or Rejecting a Candidate Mask

A candidate mask is accepted only if all selected conditions are satisfied:

$$
\operatorname{Accept}(\hat{M}_{k+d})=
\begin{cases}
1, & \text{if the candidate passes the reliability checks}\\
0, & \text{otherwise}
\end{cases}
$$

If accepted, propagation continues:

$$M_k\leftarrow\hat{M}_{k+d}$$

If rejected, propagation stops in that direction.

The forward and backward directions stop independently. For example, propagation may stop after three slices in the negative direction but continue for five slices in the positive direction.

## 5.5 Preventing Error Accumulation

Bidirectional propagation is an iterative process. An inaccurate mask can generate inaccurate prompts for the next slice:

$$
\text{mask error}
\rightarrow
\text{prompt error}
\rightarrow
\text{larger next-mask error}
$$

This is known as **error accumulation** or **propagation drift**.

The stopping criteria reduce this risk, but they cannot guarantee that every accepted mask is correct. The learner must still inspect the propagation sequence visually.

A reliable sequence should usually show:

- gradual changes in mask size;
- smooth movement of the target center;
- reasonable overlap between neighboring masks;
- disappearance of the mask near the ends of the target.

Sudden expansion, fragmentation, or movement toward another structure may indicate propagation drift.

## 5.6 Recording the Propagation History

For each accepted slice, the notebook will record:

- slice index;
- propagation direction;
- mask area;
- area ratio;
- adjacent-mask IoU;
- centroid displacement;
- stopping status or reason.

This information creates a propagation trace that helps the learner understand why a mask was accepted or why propagation stopped.

The accepted masks are stored according to their slice indices:

$$\mathcal{M}=\{M_k\mid k\in\mathcal{K}_{\text{accepted}}\}$$

where $\mathcal{K}_{\text{accepted}}$ is the set of slices reached by forward and backward propagation.

## 5.7 From Propagated Slices to a 3D Result

After both directions finish, the accepted masks form a stack of target cross-sections:

$$
\mathcal{V}(k,x,y)=
\begin{cases}
1, & (x,y)\in M_k\\
0, & \text{otherwise}
\end{cases}
$$

This stack is the initial volumetric segmentation produced by propagation.

Part 5 focuses on generating and evaluating this propagation sequence. Part 6 will organize the masks in the original tomogram coordinate system, inspect the result from multiple planes, and perform optional correction and cleanup.

## What You Will Do in Task 5

You will:

1. implement one reusable propagation step;
2. start from `initial_2d_mask`;
3. propagate toward increasing slice indices;
4. propagate toward decreasing slice indices;
5. evaluate every candidate mask using continuity checks;
6. stop each direction when a candidate becomes unreliable;
7. visualize the accepted masks across slices;
8. inspect the propagation history and stopping reasons.

## Task 5 — Explore Bidirectional Mask Propagation

### Goal

In this task, you will extend the one-step prediction from Task 4 into an iterative bidirectional propagation process.

Starting from the accepted seed mask, the system will:

1. generate automatic foreground and background prompts;
2. transfer them to a neighboring slice;
3. predict a candidate mask;
4. evaluate the continuity of the candidate;
5. accept the mask and continue, or reject it and stop.

The process is performed independently in both directions:

$$ s\rightarrow s+1\rightarrow s+2\rightarrow\cdots$$ and $$ s\rightarrow s-1\rightarrow s-2\rightarrow\cdots $$

### Your Tasks

1. Run bidirectional propagation using the default parameters.
2. Inspect whether the accepted masks continue to follow the same target.
3. Find where forward and backward propagation stop.
4. Use the propagation traces and history table to identify the stopping reason.
5. Relax the stopping criteria and observe whether propagation drift occurs.
6. Apply stricter criteria and observe whether propagation stops too early.
7. Select a reasonable balance between propagation range and stability.

### Parameters to Explore

- `MAX_STEPS`: limits the maximum number of slices reached in each direction.
- `MIN_MASK_AREA`: rejects extremely small masks.
- `MIN_AREA_RATIO` and `MAX_AREA_RATIO`: limit sudden mask-size changes.
- `MIN_ADJACENT_IOU`: requires spatial overlap between neighboring masks.
- `MAX_CENTROID_SHIFT`: limits movement of the mask center.
- `PROP_MASK_THRESHOLD`: controls the size of each predicted mask.

### Expected Output

The task produces:

1. a montage of the seed, accepted masks, and rejected candidates;
2. propagation traces for mask area, area ratio, IoU, and centroid displacement;
3. a history table explaining why each candidate was accepted or rejected;
4. a dictionary of accepted masks for later 3D reconstruction.

Remember that these continuity checks do not measure segmentation accuracy. They only help detect unstable changes when no ground-truth mask is available.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

from scipy.ndimage import (
    binary_dilation,
    binary_erosion,
    distance_transform_edt,
    gaussian_filter,
)
from skimage.measure import label
from skimage.segmentation import random_walker


def calculate_mask_centroid(mask):
    """Return the mask centroid as [x, y]."""
    y_coordinates, x_coordinates = np.nonzero(mask)

    if len(x_coordinates) == 0:
        return np.array([np.nan, np.nan])

    return np.array([
        x_coordinates.mean(),
        y_coordinates.mean(),
    ])


def predict_adjacent_mask(
    volume,
    plane,
    current_slice_index,
    current_mask,
    direction,
    core_erosion,
    background_gap,
    background_width,
    smoothing_sigma,
    random_walker_beta,
    mask_threshold,
):
    """Generate automatic prompts and predict one neighboring mask."""

    next_slice_index = current_slice_index + direction
    next_image = extract_real_slice(volume, plane, next_slice_index)

    current_mask = current_mask.astype(bool)

    # Extract a conservative foreground core.
    if core_erosion > 0:
        foreground_core = binary_erosion(
            current_mask,
            iterations=core_erosion,
        )
    else:
        foreground_core = current_mask.copy()

    # Preserve at least one foreground prompt if erosion removes the mask.
    distance_inside = distance_transform_edt(current_mask)
    positive_y, positive_x = np.unravel_index(
        np.argmax(distance_inside),
        distance_inside.shape,
    )

    if not foreground_core.any():
        foreground_core[positive_y, positive_x] = True

    # Create a background ring with an uncertain gap near the boundary.
    if background_gap > 0:
        background_inner = binary_dilation(
            current_mask,
            iterations=background_gap,
        )
    else:
        background_inner = current_mask.copy()

    background_outer = binary_dilation(
        current_mask,
        iterations=background_gap + background_width,
    )
    background_ring = background_outer & ~background_inner

    # Build the automatic prompt map:
    # 0 = unknown, 1 = background, and 2 = foreground.
    automatic_seed_labels = np.zeros(
        next_image.shape,
        dtype=np.uint8,
    )

    automatic_seed_labels[:2, :] = 1
    automatic_seed_labels[-2:, :] = 1
    automatic_seed_labels[:, :2] = 1
    automatic_seed_labels[:, -2:] = 1

    automatic_seed_labels[background_ring] = 1
    automatic_seed_labels[foreground_core] = 2

    # Predict a foreground score map on the neighboring slice.
    filtered_next_image = gaussian_filter(
        next_image,
        sigma=smoothing_sigma,
    )

    probability_maps = random_walker(
        filtered_next_image,
        automatic_seed_labels,
        beta=random_walker_beta,
        mode="bf",
        return_full_prob=True,
    )

    next_score_map = probability_maps[1]
    next_candidate_mask = next_score_map >= mask_threshold

    # Retain only components connected to the transferred foreground core.
    component_map = label(next_candidate_mask)

    connected_component_ids = np.unique(
        component_map[foreground_core]
    )
    connected_component_ids = connected_component_ids[
        connected_component_ids != 0
    ]

    next_candidate_mask = np.isin(
        component_map,
        connected_component_ids,
    )

    # Calculate continuity measurements.
    current_area = int(current_mask.sum())
    candidate_area = int(next_candidate_mask.sum())

    area_ratio = (
        candidate_area / current_area
        if current_area > 0
        else 0.0
    )

    intersection = int(
        np.logical_and(current_mask, next_candidate_mask).sum()
    )
    union = int(
        np.logical_or(current_mask, next_candidate_mask).sum()
    )

    adjacent_iou = intersection / union if union > 0 else 0.0

    current_centroid = calculate_mask_centroid(current_mask)
    candidate_centroid = calculate_mask_centroid(next_candidate_mask)

    if candidate_area > 0:
        centroid_shift = float(
            np.linalg.norm(candidate_centroid - current_centroid)
        )
    else:
        centroid_shift = float("inf")

    return {
        "slice_index": next_slice_index,
        "image": next_image,
        "mask": next_candidate_mask,
        "score_map": next_score_map,
        "foreground_core": foreground_core,
        "background_ring": background_ring,
        "mask_area": candidate_area,
        "area_ratio": area_ratio,
        "adjacent_iou": adjacent_iou,
        "centroid_shift": centroid_shift,
    }


def propagate_one_direction(
    volume,
    plane,
    seed_slice_index,
    seed_mask,
    direction,
    max_steps,
    core_erosion,
    background_gap,
    background_width,
    smoothing_sigma,
    random_walker_beta,
    mask_threshold,
    min_mask_area,
    min_area_ratio,
    max_area_ratio,
    min_adjacent_iou,
    max_centroid_shift,
):
    """Propagate from the seed mask in one direction."""

    plane_size = {
        "XY": volume.shape[0],
        "XZ": volume.shape[1],
        "YZ": volume.shape[2],
    }[plane]

    direction_name = "Forward" if direction == 1 else "Backward"

    current_slice_index = seed_slice_index
    current_mask = seed_mask.astype(bool).copy()

    history = []
    stop_reason = f"Reached MAX_STEPS = {max_steps}"

    for step in range(1, max_steps + 1):
        next_slice_index = current_slice_index + direction

        if not 0 <= next_slice_index < plane_size:
            stop_reason = "Reached the tomogram boundary"
            break

        result = predict_adjacent_mask(
            volume=volume,
            plane=plane,
            current_slice_index=current_slice_index,
            current_mask=current_mask,
            direction=direction,
            core_erosion=core_erosion,
            background_gap=background_gap,
            background_width=background_width,
            smoothing_sigma=smoothing_sigma,
            random_walker_beta=random_walker_beta,
            mask_threshold=mask_threshold,
        )

        rejection_reasons = []

        if result["mask_area"] == 0:
            rejection_reasons.append("Empty candidate mask")
        else:
            if result["mask_area"] < min_mask_area:
                rejection_reasons.append("Mask area below minimum")

            if result["area_ratio"] < min_area_ratio:
                rejection_reasons.append("Area ratio too small")

            if result["area_ratio"] > max_area_ratio:
                rejection_reasons.append("Area ratio too large")

            if result["adjacent_iou"] < min_adjacent_iou:
                rejection_reasons.append("Adjacent IoU too low")

            if result["centroid_shift"] > max_centroid_shift:
                rejection_reasons.append("Centroid shift too large")

        accepted = len(rejection_reasons) == 0

        result["step"] = step
        result["direction"] = direction_name
        result["accepted"] = accepted
        result["reason"] = (
            "Accepted"
            if accepted
            else "; ".join(rejection_reasons)
        )

        history.append(result)

        if not accepted:
            stop_reason = result["reason"]
            break

        # The accepted mask becomes the reference for the next step.
        current_slice_index = result["slice_index"]
        current_mask = result["mask"]

    return history, stop_reason

In [ ]:
# Automatic prompt parameters
PROP_CORE_EROSION = 2  # WriteYourCodeHere: increase for a more conservative foreground prompt.
PROP_BACKGROUND_GAP = 3  # WriteYourCodeHere: preserve uncertain space near the boundary.
PROP_BACKGROUND_WIDTH = 3  # WriteYourCodeHere: control the background-ring width.
PROP_MASK_THRESHOLD = MASK_THRESHOLD  # WriteYourCodeHere: increase for smaller predicted masks.

# Propagation range
MAX_STEPS = 8  # WriteYourCodeHere: maximum number of slices in each direction.

# Stopping criteria
MIN_MASK_AREA = 10  # WriteYourCodeHere: reject extremely small masks.
MIN_AREA_RATIO = 0.35  # WriteYourCodeHere: minimum allowed area ratio.
MAX_AREA_RATIO = 2.00  # WriteYourCodeHere: maximum allowed area ratio.
MIN_ADJACENT_IOU = 0.15  # WriteYourCodeHere: minimum overlap with the previous mask.
MAX_CENTROID_SHIFT = 8.0  # WriteYourCodeHere: maximum center movement in pixels.

if MAX_STEPS < 1:
    raise ValueError("MAX_STEPS must be at least 1.")

if MIN_MASK_AREA < 1:
    raise ValueError("MIN_MASK_AREA must be at least 1.")

if not 0 <= MIN_AREA_RATIO <= MAX_AREA_RATIO:
    raise ValueError("Check the minimum and maximum area ratios.")

if not 0 <= MIN_ADJACENT_IOU <= 1:
    raise ValueError("MIN_ADJACENT_IOU must be between 0 and 1.")

if MAX_CENTROID_SHIFT < 0:
    raise ValueError("MAX_CENTROID_SHIFT must be non-negative.")

if not 0 < PROP_MASK_THRESHOLD < 1:
    raise ValueError("PROP_MASK_THRESHOLD must be between 0 and 1.")


common_propagation_parameters = {
    "volume": real_tomogram_display,
    "plane": SEED_PLANE,
    "seed_slice_index": SEED_SLICE,
    "seed_mask": initial_2d_mask,
    "max_steps": MAX_STEPS,
    "core_erosion": PROP_CORE_EROSION,
    "background_gap": PROP_BACKGROUND_GAP,
    "background_width": PROP_BACKGROUND_WIDTH,
    "smoothing_sigma": SMOOTHING_SIGMA,
    "random_walker_beta": RANDOM_WALKER_BETA,
    "mask_threshold": PROP_MASK_THRESHOLD,
    "min_mask_area": MIN_MASK_AREA,
    "min_area_ratio": MIN_AREA_RATIO,
    "max_area_ratio": MAX_AREA_RATIO,
    "min_adjacent_iou": MIN_ADJACENT_IOU,
    "max_centroid_shift": MAX_CENTROID_SHIFT,
}

forward_history, forward_stop_reason = propagate_one_direction(
    direction=1,
    **common_propagation_parameters,
)

backward_history, backward_stop_reason = propagate_one_direction(
    direction=-1,
    **common_propagation_parameters,
)

# Preserve the seed mask and all accepted propagated masks.
propagated_masks = {
    SEED_SLICE: initial_2d_mask.astype(bool).copy()
}

for record in backward_history + forward_history:
    if record["accepted"]:
        propagated_masks[record["slice_index"]] = record["mask"].copy()

# Sort the history for later visualization and analysis.
propagation_history = sorted(
    backward_history + forward_history,
    key=lambda record: record["slice_index"],
)

accepted_slice_indices = sorted(propagated_masks.keys())

propagation_stop_reasons = {
    "Backward": backward_stop_reason,
    "Forward": forward_stop_reason,
}

print(f"Seed slice: {SEED_SLICE}")
print(f"Accepted slice range: {accepted_slice_indices[0]}–{accepted_slice_indices[-1]}")
print(f"Total accepted slices, including seed: {len(accepted_slice_indices)}")
print()
print(f"Backward stop: {backward_stop_reason}")
print(f"Forward stop: {forward_stop_reason}")

In [ ]:
# ---------------------------------------------------------
# 1. Bidirectional propagation montage
# ---------------------------------------------------------

display_records = [
    {
        "slice_index": SEED_SLICE,
        "mask": initial_2d_mask.astype(bool),
        "direction": "Seed",
        "accepted": True,
        "reason": "Manual seed mask",
    }
]

display_records.extend(backward_history)
display_records.extend(forward_history)

display_records = sorted(
    display_records,
    key=lambda record: record["slice_index"],
)

number_of_columns = min(5, len(display_records))
number_of_rows = int(
    np.ceil(len(display_records) / number_of_columns)
)

fig, axes = plt.subplots(
    number_of_rows,
    number_of_columns,
    figsize=(3.4 * number_of_columns, 3.4 * number_of_rows),
)

axes = np.atleast_1d(axes).ravel()

for axis, record in zip(axes, display_records):
    slice_image = extract_real_slice(
        real_tomogram_display,
        SEED_PLANE,
        record["slice_index"],
    )

    axis.imshow(slice_image, cmap="gray", origin="lower")

    if record["direction"] == "Seed":
        overlay_cmap = "winter"
        status = "Seed"
    elif record["accepted"]:
        overlay_cmap = "autumn"
        status = "Accepted"
    else:
        overlay_cmap = "Reds"
        status = "Rejected"

    display_mask = record["mask"].astype(bool)

    axis.imshow(
        np.ma.masked_where(~display_mask, display_mask),
        cmap=overlay_cmap,
        alpha=0.45,
        origin="lower",
    )

    if display_mask.any():
        contour_color = (
            "cyan"
            if record["direction"] == "Seed"
            else "yellow"
            if record["accepted"]
            else "red"
        )

        axis.contour(
            display_mask,
            levels=[0.5],
            colors=contour_color,
            linewidths=1.2,
            origin="lower",
        )

    axis.set_title(
        f"Slice {record['slice_index']}\n"
        f"{record['direction']} | {status}",
        fontsize=10,
    )
    axis.axis("off")

for axis in axes[len(display_records):]:
    axis.axis("off")

plt.suptitle(
    "Bidirectional Propagation Montage",
    fontsize=14,
    y=1.02,
)
plt.tight_layout()
plt.show()


# ---------------------------------------------------------
# 2. Propagation traces
# ---------------------------------------------------------

fig, trace_axes = plt.subplots(
    2,
    2,
    figsize=(13, 9),
)

direction_styles = [
    (backward_history, "Backward", "tab:blue"),
    (forward_history, "Forward", "tab:orange"),
]


def plot_history_metric(axis, metric_name, ylabel):
    for history, direction_name, color in direction_styles:
        if len(history) == 0:
            continue

        slice_indices = np.array([
            record["slice_index"]
            for record in history
        ])

        metric_values = np.array([
            record[metric_name]
            for record in history
        ], dtype=float)

        finite_values = np.isfinite(metric_values)

        axis.plot(
            slice_indices[finite_values],
            metric_values[finite_values],
            color=color,
            linewidth=1.5,
            alpha=0.8,
            label=direction_name,
        )

        for record in history:
            value = float(record[metric_name])

            if not np.isfinite(value):
                continue

            marker = "o" if record["accepted"] else "x"
            marker_color = "green" if record["accepted"] else "red"

            axis.scatter(
                record["slice_index"],
                value,
                color=marker_color,
                marker=marker,
                s=65,
                zorder=3,
            )

    axis.set_xlabel("Slice Index")
    axis.set_ylabel(ylabel)
    axis.grid(alpha=0.25)


plot_history_metric(
    trace_axes[0, 0],
    "mask_area",
    "Mask Area (pixels)",
)
trace_axes[0, 0].axhline(
    MIN_MASK_AREA,
    color="red",
    linestyle="--",
    alpha=0.7,
)
trace_axes[0, 0].set_title("Mask Area")

plot_history_metric(
    trace_axes[0, 1],
    "area_ratio",
    "Area Ratio",
)
trace_axes[0, 1].axhline(
    MIN_AREA_RATIO,
    color="red",
    linestyle="--",
    alpha=0.7,
)
trace_axes[0, 1].axhline(
    MAX_AREA_RATIO,
    color="red",
    linestyle="--",
    alpha=0.7,
)
trace_axes[0, 1].set_title("Adjacent-Slice Area Ratio")

plot_history_metric(
    trace_axes[1, 0],
    "adjacent_iou",
    "Adjacent-Mask IoU",
)
trace_axes[1, 0].axhline(
    MIN_ADJACENT_IOU,
    color="red",
    linestyle="--",
    alpha=0.7,
)
trace_axes[1, 0].set_ylim(0, 1)
trace_axes[1, 0].set_title("Adjacent-Mask IoU")

plot_history_metric(
    trace_axes[1, 1],
    "centroid_shift",
    "Centroid Shift (pixels)",
)
trace_axes[1, 1].axhline(
    MAX_CENTROID_SHIFT,
    color="red",
    linestyle="--",
    alpha=0.7,
)
trace_axes[1, 1].set_title("Centroid Displacement")

handles, labels = trace_axes[0, 0].get_legend_handles_labels()
if handles:
    fig.legend(
        handles,
        labels,
        loc="upper center",
        ncol=2,
    )

plt.suptitle(
    "Propagation Continuity Traces\n"
    "Green circles: accepted | Red crosses: rejected | "
    "Red dashed lines: thresholds",
    fontsize=13,
)
plt.tight_layout(rect=[0, 0, 1, 0.93])
plt.show()


# ---------------------------------------------------------
# 3. Propagation history table
# ---------------------------------------------------------

print("Propagation History")
print("-" * 118)

header = (
    f"{'Slice':>6} "
    f"{'Direction':>10} "
    f"{'Area':>8} "
    f"{'Ratio':>9} "
    f"{'IoU':>9} "
    f"{'Shift':>9} "
    f"{'Status':>10}  "
    f"Reason"
)

print(header)
print("-" * 118)

for record in propagation_history:
    shift_text = (
        f"{record['centroid_shift']:.3f}"
        if np.isfinite(record["centroid_shift"])
        else "inf"
    )

    status = "Accepted" if record["accepted"] else "Rejected"

    print(
        f"{record['slice_index']:>6} "
        f"{record['direction']:>10} "
        f"{record['mask_area']:>8} "
        f"{record['area_ratio']:>9.3f} "
        f"{record['adjacent_iou']:>9.3f} "
        f"{shift_text:>9} "
        f"{status:>10}  "
        f"{record['reason']}"
    )

print("-" * 118)
print(f"Backward stop: {backward_stop_reason}")
print(f"Forward stop:  {forward_stop_reason}")

# Part 6 — Reconstructing and Inspecting the 3D Mask

In Part 5, the initial 2D mask was propagated to neighboring slices in both directions, and each accepted result was stored as `propagated_masks[slice_index] = mask`. These masks describe the target on individual slices but do not yet form a complete volumetric segmentation. In this part, the accepted 2D masks will be placed back into the original tomogram coordinate system to reconstruct and inspect a 3D binary mask.

## 6.1 From 2D Masks to a 3D Volume

First, an empty binary array is created with the same shape as the tomogram:

$$V_M\in\{0,1\}^{Z\times Y\times X}$$

A value of 1 represents a foreground voxel, while 0 represents the background. Each propagated mask is inserted according to the selected propagation plane.

For propagation through XY slices:

$$V_M[k,:,:]=M_k$$

For propagation through XZ slices:

$$V_M[:,k,:]=M_k$$

For propagation through YZ slices:

$$V_M[:,:,k]=M_k$$

Here, $k$ is the slice index and $M_k$ is the corresponding accepted 2D mask. Slices without an accepted mask remain empty. The reconstructed mask must satisfy:

$$\operatorname{shape}(V_M)=\operatorname{shape}(V_{\text{tomogram}})$$

This guarantees that every mask voxel is aligned with the corresponding voxel in the original tomogram.

## 6.2 Why Inspect the Mask in Multiple Planes?

The masks were predicted slice by slice in one propagation plane. A result may appear smooth in that plane while containing errors that are easier to detect from another direction. The reconstructed volume should therefore be inspected using three orthogonal views:

$$V_M[z,:,:]\quad\text{for the XY view}$$

$$V_M[:,y,:]\quad\text{for the XZ view}$$

$$V_M[:,:,x]\quad\text{for the YZ view}$$

These views can reveal sudden changes in target width, disconnected regions, sharp boundary jumps, isolated fragments, or masks that extend into nearby structures. A reliable 3D segmentation should remain spatially coherent in all three planes.

## 6.3 Tomogram–Mask Overlay

A binary mask is easier to evaluate when displayed over the corresponding tomogram slice. For an image slice $I$ and mask $M$, the overlay can be represented as:

$$I_{\text{overlay}}=(1-\alpha)I+\alpha C(M)$$

Here, $\alpha$ controls the transparency and $C(M)$ assigns a display color to the foreground region. The overlay helps determine whether the mask continues to follow the same biological structure across neighboring slices. Therefore, the mask should be evaluated using both its boundary and the underlying image information.

## 6.4 Orthogonal Inspection Point

A 3D coordinate $(z_c,y_c,x_c)$ defines three intersecting views: an XY slice at $z=z_c$, an XZ slice at $y=y_c$, and a YZ slice at $x=x_c$. A useful initial inspection point is the centroid of the reconstructed foreground:

$$\mathbf{c}_{3D}=\frac{1}{N}\sum_{V_M(z,y,x)=1}(z,y,x)$$

Here, $N$ is the total number of foreground voxels. Moving this coordinate allows different parts of the segmented target and its surrounding region to be inspected.

## 6.5 Orthogonal Mask Projections

The overall spatial extent of the target can also be summarized using maximum projections:

$$P_{XY}(y,x)=\max_z V_M(z,y,x)$$

$$P_{XZ}(z,x)=\max_y V_M(z,y,x)$$

$$P_{YZ}(z,y)=\max_x V_M(z,y,x)$$

Because the mask is binary, a projected pixel is foreground if at least one foreground voxel exists along the corresponding direction. These projections provide a compact overview of the target shape, but they remove depth information and should be used together with slice-by-slice inspection.

## 6.6 Three-Dimensional Connected Components

Propagation may occasionally create isolated foreground fragments. A 3D connected-component analysis divides the reconstructed mask into spatially connected regions:

$$V_M=C_1\cup C_2\cup\cdots\cup C_n$$

For a single-target segmentation, the component connected to the original seed mask is normally the most relevant:

$$V_M^{\text{clean}}=C_{\text{seed}}$$

Retaining the seed-connected component can remove unrelated fragments while preserving the structure selected by the original prompt. However, this cleanup should remain optional because a real target may appear disconnected if an intermediate slice was segmented incorrectly. The result must therefore still be inspected visually.

## 6.7 Identifying Suspicious Slices

The continuity measurements calculated in Part 5 can help identify slices that require closer inspection. A slice may be suspicious if it has a low adjacent-mask IoU, a large area change, a large centroid displacement, or visible disagreement between the mask and tomogram. Passing the automatic stopping criteria does not guarantee that a mask is correct because these criteria only reject strong discontinuities. Final validation still requires multi-plane visual inspection.

## 6.8 Local Mask Correction

If one propagated mask is inaccurate, the selected slice can be segmented again using new positive and optional negative point prompts:

$$I_k+P_k\rightarrow S_k\rightarrow M_k^{\text{corrected}}$$

If the candidate mask is more accurate, it can replace the propagated result:

$$M_k\leftarrow M_k^{\text{corrected}}$$

The corresponding plane in the 3D volume must also be updated. For example, an XY correction is inserted using:

$$V_M[k,:,:]\leftarrow M_k^{\text{corrected}}$$

Equivalent insertion rules apply to XZ and YZ slices. This part uses local correction, meaning that only the selected slice is replaced. Re-propagating from the corrected slice could update its neighbors, but that is outside the main workflow of this tutorial.

## 6.9 Result of Part 6

After reconstruction, multi-plane inspection, optional connected-component cleanup, and local correction, the finalized segmentation will be stored as `finalized_mask_volume_3d`. This binary 3D mask remains spatially aligned with the original tomogram.

The workflow of this part is:

**Propagated 2D masks → coordinate-aware insertion → reconstructed 3D mask → multi-plane inspection → optional cleanup and correction → finalized 3D segmentation**

In Part 7, the finalized 3D mask will be used to calculate quantitative properties such as voxel count, physical volume, centroid, bounding box, and slice span.

## Task 6 — Reconstruct, Inspect, and Correct the 3D Mask

### Goal

In this task, you will reconstruct a complete 3D binary mask from the accepted 2D masks generated in Task 5.

You will then inspect the result from the XY, XZ, and YZ planes, identify possible discontinuities, optionally retain the 3D component connected to the seed mask, and test a local correction on one propagated slice.

### Your Tasks

1. Insert every accepted 2D mask into its original position in a 3D array.
2. Verify that the reconstructed mask has the same shape as the tomogram.
3. Compare the original mask with the seed-connected 3D component.
4. Inspect the mask from the XY, XZ, and YZ planes.
5. Examine the three orthogonal mask projections.
6. Use the propagation history to identify suspicious slices.
7. Select one slice and define positive and optional negative correction points.
8. Compare the propagated mask with the locally corrected mask.
9. Accept the correction only if it improves alignment with the target.

### Parameters to Explore

- `USE_SEED_CONNECTED_COMPONENT`: enables or disables 3D component cleanup.
- `VIEW_Z`, `VIEW_Y`, and `VIEW_X`: select the orthogonal inspection position.
- `CORRECTION_SLICE`: selects the propagated slice to correct.
- `POSITIVE_POINTS`: marks locations inside the target.
- `NEGATIVE_POINTS`: marks locations outside the target.
- `CORRECTION_THRESHOLD`: controls the size of the corrected mask.
- `ACCEPT_LOCAL_CORRECTION`: determines whether the candidate correction replaces the propagated mask.

Point coordinates must be entered as:

```python
(x, y)

In [ ]:
import numpy as np
from scipy.ndimage import generate_binary_structure, label as ndi_label

def extract_mask_plane(array_3d, plane, index):
    """Extract an XY, XZ, or YZ plane from a 3D array."""
    plane = plane.upper()
    if plane == "XY":
        return array_3d[index, :, :]
    if plane == "XZ":
        return array_3d[:, index, :]
    if plane == "YZ":
        return array_3d[:, :, index]
    raise ValueError("plane must be 'XY', 'XZ', or 'YZ'.")

def insert_mask_plane(array_3d, plane, index, mask):
    """Insert a 2D mask into its original position in a 3D array."""
    plane = plane.upper()
    expected_shape = {"XY": array_3d.shape[1:], "XZ": (array_3d.shape[0], array_3d.shape[2]), "YZ": array_3d.shape[:2]}[plane]
    if mask.shape != expected_shape:
        raise ValueError(f"Mask shape {mask.shape} does not match the expected {plane} shape {expected_shape}.")
    if plane == "XY":
        array_3d[index, :, :] = mask
    elif plane == "XZ":
        array_3d[:, index, :] = mask
    else:
        array_3d[:, :, index] = mask

# Insert all accepted 2D masks into a full 3D array.
mask_volume_3d = np.zeros(real_tomogram_display.shape, dtype=bool)
for slice_index, mask in propagated_masks.items():
    insert_mask_plane(mask_volume_3d, SEED_PLANE, int(slice_index), mask.astype(bool))

if mask_volume_3d.shape != real_tomogram_display.shape:
    raise ValueError("The reconstructed mask and tomogram must have the same shape.")

if not mask_volume_3d.any():
    raise ValueError("The reconstructed 3D mask is empty.")

# Identify 3D connected components using 26-connectivity.
connectivity_3d = generate_binary_structure(3, 3)
component_labels_3d, number_of_components = ndi_label(mask_volume_3d, structure=connectivity_3d)
component_sizes = np.bincount(component_labels_3d.ravel())[1:]

# Find the component with the greatest overlap with the original seed mask.
seed_component_slice = extract_mask_plane(component_labels_3d, SEED_PLANE, SEED_SLICE)
seed_component_ids = seed_component_slice[initial_2d_mask.astype(bool)]
seed_component_ids = seed_component_ids[seed_component_ids > 0]

if len(seed_component_ids) == 0:
    raise ValueError("No 3D component overlaps the original seed mask.")

unique_seed_ids, seed_overlap_counts = np.unique(seed_component_ids, return_counts=True)
seed_component_id = int(unique_seed_ids[np.argmax(seed_overlap_counts)])

USE_SEED_CONNECTED_COMPONENT = True  # WriteYourCodeHere: set False to retain every 3D component.

if USE_SEED_CONNECTED_COMPONENT:
    cleaned_mask_volume_3d = component_labels_3d == seed_component_id
else:
    cleaned_mask_volume_3d = mask_volume_3d.copy()

# Create synchronized 2D and 3D results for later correction.
corrected_propagated_masks = {int(index): extract_mask_plane(cleaned_mask_volume_3d, SEED_PLANE, int(index)).copy() for index in propagated_masks}
accepted_slice_indices = sorted(index for index, mask in corrected_propagated_masks.items() if mask.any())
finalized_mask_volume_3d = cleaned_mask_volume_3d.copy()

print(f"Tomogram shape: {real_tomogram_display.shape}")
print(f"3D mask shape: {mask_volume_3d.shape}")
print(f"Accepted 2D masks: {len(propagated_masks)}")
print(f"Original foreground voxels: {int(mask_volume_3d.sum())}")
print(f"Detected 3D components: {number_of_components}")
print(f"Component sizes: {component_sizes.tolist()}")
print(f"Seed-connected component: {seed_component_id}")
print(f"Foreground voxels after cleanup: {int(cleaned_mask_volume_3d.sum())}")
print(f"Final slice range: {accepted_slice_indices[0]}–{accepted_slice_indices[-1]}")

In [ ]:
import matplotlib.pyplot as plt

mask_coordinates_zyx = np.argwhere(finalized_mask_volume_3d)
if len(mask_coordinates_zyx) == 0:
    raise ValueError("The finalized 3D mask is empty.")

# Start at the 3D mask centroid. Replace these values to inspect another location.
VIEW_Z, VIEW_Y, VIEW_X = np.rint(mask_coordinates_zyx.mean(axis=0)).astype(int)  # WriteYourCodeHere
VIEW_Z = int(np.clip(VIEW_Z, 0, real_tomogram_display.shape[0] - 1))
VIEW_Y = int(np.clip(VIEW_Y, 0, real_tomogram_display.shape[1] - 1))
VIEW_X = int(np.clip(VIEW_X, 0, real_tomogram_display.shape[2] - 1))

xy_image, xy_mask = real_tomogram_display[VIEW_Z, :, :], finalized_mask_volume_3d[VIEW_Z, :, :]
xz_image, xz_mask = real_tomogram_display[:, VIEW_Y, :], finalized_mask_volume_3d[:, VIEW_Y, :]
yz_image, yz_mask = real_tomogram_display[:, :, VIEW_X], finalized_mask_volume_3d[:, :, VIEW_X]

def display_mask_overlay(axis, image, mask, title, horizontal_coordinate, vertical_coordinate, horizontal_label, vertical_label):
    """Display a tomogram slice, mask overlay, and orthogonal intersection."""
    axis.imshow(image, cmap="gray", origin="lower")
    axis.imshow(np.ma.masked_where(~mask, mask), cmap="autumn", alpha=0.45, origin="lower")
    if mask.any():
        axis.contour(mask, levels=[0.5], colors="yellow", linewidths=1.0, origin="lower")
    axis.axvline(horizontal_coordinate, color="cyan", linestyle="--", linewidth=0.9)
    axis.axhline(vertical_coordinate, color="cyan", linestyle="--", linewidth=0.9)
    axis.set_title(title)
    axis.set_xlabel(horizontal_label)
    axis.set_ylabel(vertical_label)

xy_projection = finalized_mask_volume_3d.max(axis=0)
xz_projection = finalized_mask_volume_3d.max(axis=1)
yz_projection = finalized_mask_volume_3d.max(axis=2)

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
display_mask_overlay(axes[0, 0], xy_image, xy_mask, f"XY View at z = {VIEW_Z}", VIEW_X, VIEW_Y, "x", "y")
display_mask_overlay(axes[0, 1], xz_image, xz_mask, f"XZ View at y = {VIEW_Y}", VIEW_X, VIEW_Z, "x", "z")
display_mask_overlay(axes[0, 2], yz_image, yz_mask, f"YZ View at x = {VIEW_X}", VIEW_Y, VIEW_Z, "y", "z")

projections = [xy_projection, xz_projection, yz_projection]
projection_titles = ["XY Mask Projection", "XZ Mask Projection", "YZ Mask Projection"]
projection_labels = [("x", "y"), ("x", "z"), ("y", "z")]

for axis, projection, title, labels in zip(axes[1], projections, projection_titles, projection_labels):
    axis.imshow(projection, cmap="magma", origin="lower")
    if projection.any():
        axis.contour(projection, levels=[0.5], colors="cyan", linewidths=0.8, origin="lower")
    axis.set_title(title)
    axis.set_xlabel(labels[0])
    axis.set_ylabel(labels[1])

plt.suptitle(f"Orthogonal Inspection at (z, y, x) = ({VIEW_Z}, {VIEW_Y}, {VIEW_X})", fontsize=14)
plt.tight_layout()
plt.show()

# List accepted slices with the lowest adjacent-mask IoU for closer inspection.
accepted_history_records = [record for record in propagation_history if record["accepted"]]
suspicious_records = sorted(accepted_history_records, key=lambda record: record["adjacent_iou"])[:5]

print("Suggested slices for closer inspection")
print("-" * 76)
print(f"{'Slice':>6} {'Direction':>10} {'Area':>8} {'Ratio':>9} {'IoU':>9} {'Shift':>9}")
print("-" * 76)

for record in suspicious_records:
    print(f"{record['slice_index']:>6} {record['direction']:>10} {record['mask_area']:>8} {record['area_ratio']:>9.3f} {record['adjacent_iou']:>9.3f} {record['centroid_shift']:>9.3f}")

print("-" * 76)
print("A low IoU indicates a strong change, but visual inspection is still required.")

In [ ]:
from scipy.ndimage import gaussian_filter
from skimage.measure import label as sk_label
from skimage.segmentation import random_walker

# By default, inspect the accepted slice with the lowest adjacent-mask IoU.
CORRECTION_SLICE = suspicious_records[0]["slice_index"] if suspicious_records else SEED_SLICE  # WriteYourCodeHere

if CORRECTION_SLICE not in corrected_propagated_masks:
    raise ValueError("CORRECTION_SLICE must be one of the propagated slice indices.")

correction_image = extract_real_slice(real_tomogram_display, SEED_PLANE, CORRECTION_SLICE)
propagated_mask_before_correction = corrected_propagated_masks[CORRECTION_SLICE].copy()
current_mask_coordinates = np.argwhere(propagated_mask_before_correction)

if len(current_mask_coordinates) == 0:
    raise ValueError("The selected propagated mask is empty.")

default_y, default_x = np.rint(current_mask_coordinates.mean(axis=0)).astype(int)

# Coordinates are entered as (x, y). Add more points when necessary.
POSITIVE_POINTS = [(int(default_x), int(default_y))]  # WriteYourCodeHere
NEGATIVE_POINTS = []  # WriteYourCodeHere: for example, [(20, 30), (80, 60)].
CORRECTION_THRESHOLD = PROP_MASK_THRESHOLD  # WriteYourCodeHere
CORRECTION_BETA = RANDOM_WALKER_BETA  # WriteYourCodeHere
ACCEPT_LOCAL_CORRECTION = False  # WriteYourCodeHere: inspect the result before changing this to True.

def segment_slice_with_points(image, positive_points, negative_points, beta, smoothing_sigma, threshold):
    """Generate a candidate correction using positive and negative point prompts."""
    height, width = image.shape
    if len(positive_points) == 0:
        raise ValueError("At least one positive point is required.")
    for x, y in positive_points + negative_points:
        if not (0 <= x < width and 0 <= y < height):
            raise ValueError(f"Point {(x, y)} is outside the image.")
    point_labels = np.zeros(image.shape, dtype=np.uint8)
    point_labels[:2, :], point_labels[-2:, :] = 1, 1
    point_labels[:, :2], point_labels[:, -2:] = 1, 1
    for x, y in negative_points:
        point_labels[y, x] = 1
    for x, y in positive_points:
        point_labels[y, x] = 2
    filtered_image = gaussian_filter(image, sigma=smoothing_sigma)
    probability_maps = random_walker(filtered_image, point_labels, beta=beta, mode="bf", return_full_prob=True)
    foreground_score = probability_maps[1]
    candidate_mask = foreground_score >= threshold
    component_map = sk_label(candidate_mask)
    positive_component_ids = np.unique([component_map[y, x] for x, y in positive_points])
    positive_component_ids = positive_component_ids[positive_component_ids > 0]
    candidate_mask = np.isin(component_map, positive_component_ids)
    return candidate_mask, foreground_score, point_labels

corrected_candidate_mask, correction_score_map, correction_point_labels = segment_slice_with_points(correction_image, POSITIVE_POINTS, NEGATIVE_POINTS, CORRECTION_BETA, SMOOTHING_SIGMA, CORRECTION_THRESHOLD)

intersection = np.logical_and(propagated_mask_before_correction, corrected_candidate_mask).sum()
union = np.logical_or(propagated_mask_before_correction, corrected_candidate_mask).sum()
correction_iou = intersection / union if union > 0 else 0.0

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

axes[0].imshow(correction_image, cmap="gray", origin="lower")
axes[0].imshow(np.ma.masked_where(~propagated_mask_before_correction, propagated_mask_before_correction), cmap="autumn", alpha=0.45, origin="lower")
if propagated_mask_before_correction.any():
    axes[0].contour(propagated_mask_before_correction, levels=[0.5], colors="yellow", linewidths=1.1, origin="lower")
axes[0].set_title("Original Propagated Mask")

axes[1].imshow(correction_image, cmap="gray", origin="lower")
for x, y in POSITIVE_POINTS:
    axes[1].scatter(x, y, c="lime", marker="o", s=75, edgecolors="black")
for x, y in NEGATIVE_POINTS:
    axes[1].scatter(x, y, c="red", marker="x", s=80, linewidths=2)
axes[1].set_title("Correction Point Prompts")

axes[2].imshow(correction_image, cmap="gray", origin="lower")
axes[2].imshow(np.ma.masked_where(~corrected_candidate_mask, corrected_candidate_mask), cmap="winter", alpha=0.45, origin="lower")
if corrected_candidate_mask.any():
    axes[2].contour(corrected_candidate_mask, levels=[0.5], colors="cyan", linewidths=1.1, origin="lower")
axes[2].set_title("Candidate Corrected Mask")

for axis in axes:
    axis.set_xlabel("x")
    axis.set_ylabel("y")

plt.suptitle(f"Local Correction on {SEED_PLANE} Slice {CORRECTION_SLICE}")
plt.tight_layout()
plt.show()

print(f"Original mask area: {int(propagated_mask_before_correction.sum())} pixels")
print(f"Corrected candidate area: {int(corrected_candidate_mask.sum())} pixels")
print(f"IoU between original and candidate: {correction_iou:.3f}")

if ACCEPT_LOCAL_CORRECTION:
    corrected_propagated_masks[CORRECTION_SLICE] = corrected_candidate_mask.copy()
    insert_mask_plane(finalized_mask_volume_3d, SEED_PLANE, CORRECTION_SLICE, corrected_candidate_mask)
    accepted_slice_indices = sorted(index for index, mask in corrected_propagated_masks.items() if mask.any())
    print("The local correction was accepted and saved.")
else:
    print("The candidate was not saved. Inspect it first, then set ACCEPT_LOCAL_CORRECTION = True if appropriate.")

print(f"Finalized foreground voxels: {int(finalized_mask_volume_3d.sum())}")

# Part 7 — Quantitative Measurement of the 3D Segmentation

Part 6 produced `finalized_mask_volume_3d`, a binary mask aligned with the original tomogram. In this part, the mask will be converted into quantitative measurements describing the segmented target’s size, position, and spatial extent.

## 7.1 Voxel Count and Physical Volume

The foreground voxel count is:

$$N=\sum_{z,y,x}V_M(z,y,x)$$

If the voxel spacing is $(\Delta z,\Delta y,\Delta x)$, the physical volume represented by one voxel is:

$$V_{\text{voxel}}=\Delta z\Delta y\Delta x$$

The total target volume is therefore:

$$V_{\text{target}}=N\Delta z\Delta y\Delta x$$

The voxel spacing must be obtained from the tomogram metadata. If it is unavailable, the spacing can temporarily be set to $(1,1,1)$, but the result should then be interpreted in voxel units rather than as a true physical measurement.

## 7.2 Three-Dimensional Centroid

Let the foreground voxel coordinates be $(z_i,y_i,x_i)$. Their mean gives the centroid in voxel-index coordinates:

$$\mathbf{c}_{\text{voxel}}=\frac{1}{N}\sum_{i=1}^{N}(z_i,y_i,x_i)$$

If the origin is placed at the outer corner of voxel $(0,0,0)$, the corresponding physical centroid is:

$$\mathbf{c}_{\text{physical}}=(\mathbf{c}_{\text{voxel}}+0.5)\odot(\Delta z,\Delta y,\Delta x)$$

The addition of 0.5 moves each integer array index to the center of its voxel.

## 7.3 Three-Dimensional Bounding Box

The smallest axis-aligned bounding box containing the complete target is defined by the minimum and maximum foreground indices:

$$\mathbf{b}_{\min}=(z_{\min},y_{\min},x_{\min})$$

$$\mathbf{b}_{\max}=(z_{\max},y_{\max},x_{\max})$$

Because both boundary indices contain foreground voxels, the bounding-box size in voxels is:

$$\mathbf{d}_{\text{voxel}}=\mathbf{b}_{\max}-\mathbf{b}_{\min}+1$$

The corresponding physical dimensions are:

$$\mathbf{d}_{\text{physical}}=\mathbf{d}_{\text{voxel}}\odot(\Delta z,\Delta y,\Delta x)$$

The bounding box describes the target’s maximum extent along each axis, but it does not account for empty space inside the box.

## 7.4 Slice Span and Cross-Sectional Area

The number of foreground pixels on each slice forms a cross-sectional area profile. For XY slices:

$$N_{XY}(z)=\sum_{y,x}V_M(z,y,x)$$

The physical area on an XY slice is:

$$A_{XY}(z)=N_{XY}(z)\Delta y\Delta x$$

Similarly:

$$A_{XZ}(y)=\left(\sum_{z,x}V_M(z,y,x)\right)\Delta z\Delta x$$

$$A_{YZ}(x)=\left(\sum_{z,y}V_M(z,y,x)\right)\Delta z\Delta y$$

The first and last slices with nonzero area define the target’s slice range. The number of occupied slices should also be reported separately because gaps may exist inside this range.

## 7.5 Interpreting the Measurements

These measurements quantify the current segmentation, not necessarily the exact biological structure. Their accuracy depends on the tomogram resolution, voxel spacing, seed-mask quality, propagation thresholds, stopping rules, and any corrections applied in Part 6.

Voxel count is independent of the physical unit, but physical volume, centroid, dimensions, and cross-sectional area are meaningful only when the correct voxel spacing is provided.

## 7.6 Result of Part 7

The main results will be stored in `measurement_summary` and `area_profiles`. Together, they describe the target’s voxel count, physical volume, centroid, bounding box, occupied slice range, and cross-sectional area in all three viewing planes.

The workflow of this part is:

**Finalized 3D mask → foreground coordinates → voxel and physical measurements → area profiles → quantitative target description**

## Task 7 — Measure the Segmented Target

### Goal

In this task, you will calculate quantitative properties from `finalized_mask_volume_3d` and visualize how the target’s cross-sectional area changes through the volume.

### Your Tasks

1. Enter the tomogram voxel spacing in $(z,y,x)$ order.
2. Calculate the foreground voxel count and physical volume.
3. Determine the centroid in voxel and physical coordinates.
4. Calculate the 3D bounding box and its physical dimensions.
5. Find the occupied range and number of occupied slices along each axis.
6. Calculate the XY, XZ, and YZ cross-sectional area profiles.
7. Identify the slice with the largest cross-sectional area in each plane.
8. Interpret the measurements together with the segmentation shown in Part 6.

### Parameters to Complete

- `VOXEL_SIZE_ZYX`: voxel spacing in $(z,y,x)$ order.
- `LENGTH_UNIT`: physical length unit used by the spacing values.

If the true voxel spacing is unknown, use `(1.0, 1.0, 1.0)` and set `LENGTH_UNIT = "voxel"`. In this case, the calculated values are expressed in voxel-based units rather than true physical units.

In [ ]:
import numpy as np

VOXEL_SIZE_ZYX = (1.0, 1.0, 1.0)  # WriteYourCodeHere: replace with the tomogram spacing in (z, y, x) order.
LENGTH_UNIT = "voxel"  # WriteYourCodeHere: for example, "nm" or "Å".

measurement_mask = finalized_mask_volume_3d.astype(bool)
voxel_size_zyx = np.asarray(VOXEL_SIZE_ZYX, dtype=float)

if measurement_mask.ndim != 3:
    raise ValueError("finalized_mask_volume_3d must be a 3D array.")
if measurement_mask.shape != real_tomogram_display.shape:
    raise ValueError("The finalized mask and tomogram must have the same shape.")
if not measurement_mask.any():
    raise ValueError("The finalized 3D mask is empty.")
if voxel_size_zyx.shape != (3,) or np.any(voxel_size_zyx <= 0):
    raise ValueError("VOXEL_SIZE_ZYX must contain three positive values in (z, y, x) order.")

foreground_coordinates_zyx = np.argwhere(measurement_mask)
foreground_voxel_count = int(foreground_coordinates_zyx.shape[0])
single_voxel_volume = float(np.prod(voxel_size_zyx))
target_volume = foreground_voxel_count * single_voxel_volume

centroid_voxel_zyx = foreground_coordinates_zyx.mean(axis=0)
centroid_physical_zyx = (centroid_voxel_zyx + 0.5) * voxel_size_zyx

bbox_min_zyx = foreground_coordinates_zyx.min(axis=0)
bbox_max_zyx = foreground_coordinates_zyx.max(axis=0)
bbox_size_voxels_zyx = bbox_max_zyx - bbox_min_zyx + 1
bbox_min_physical_zyx = bbox_min_zyx * voxel_size_zyx
bbox_max_physical_zyx = (bbox_max_zyx + 1) * voxel_size_zyx
bbox_size_physical_zyx = bbox_size_voxels_zyx * voxel_size_zyx

voxel_counts_xy = measurement_mask.sum(axis=(1, 2))
voxel_counts_xz = measurement_mask.sum(axis=(0, 2))
voxel_counts_yz = measurement_mask.sum(axis=(0, 1))
areas_xy = voxel_counts_xy * voxel_size_zyx[1] * voxel_size_zyx[2]
areas_xz = voxel_counts_xz * voxel_size_zyx[0] * voxel_size_zyx[2]
areas_yz = voxel_counts_yz * voxel_size_zyx[0] * voxel_size_zyx[1]

def summarize_axis_profile(profile):
    """Return the occupied range, number of occupied slices, and maximum-profile slice."""
    occupied = np.flatnonzero(profile > 0)
    return {"first_slice": int(occupied[0]), "last_slice": int(occupied[-1]), "occupied_slices": int(len(occupied)), "range_length": int(occupied[-1] - occupied[0] + 1), "maximum_slice": int(np.argmax(profile)), "maximum_value": float(np.max(profile))}

xy_summary = summarize_axis_profile(areas_xy)
xz_summary = summarize_axis_profile(areas_xz)
yz_summary = summarize_axis_profile(areas_yz)

area_profiles = {"XY": areas_xy, "XZ": areas_xz, "YZ": areas_yz}
measurement_summary = {
    "voxel_size_zyx": tuple(voxel_size_zyx),
    "length_unit": LENGTH_UNIT,
    "foreground_voxel_count": foreground_voxel_count,
    "single_voxel_volume": single_voxel_volume,
    "target_volume": target_volume,
    "centroid_voxel_zyx": tuple(centroid_voxel_zyx),
    "centroid_physical_zyx": tuple(centroid_physical_zyx),
    "bbox_min_zyx": tuple(bbox_min_zyx),
    "bbox_max_zyx": tuple(bbox_max_zyx),
    "bbox_size_voxels_zyx": tuple(bbox_size_voxels_zyx),
    "bbox_size_physical_zyx": tuple(bbox_size_physical_zyx),
    "XY": xy_summary,
    "XZ": xz_summary,
    "YZ": yz_summary
}

print("3D Segmentation Measurements")
print("-" * 72)
print(f"Voxel spacing (z, y, x):       {tuple(voxel_size_zyx)} {LENGTH_UNIT}")
print(f"Foreground voxel count:        {foreground_voxel_count}")
print(f"Single-voxel volume:           {single_voxel_volume:.6f} {LENGTH_UNIT}³")
print(f"Target volume:                 {target_volume:.6f} {LENGTH_UNIT}³")
print(f"Centroid in voxel coordinates: ({centroid_voxel_zyx[0]:.2f}, {centroid_voxel_zyx[1]:.2f}, {centroid_voxel_zyx[2]:.2f})")
print(f"Centroid in physical units:    ({centroid_physical_zyx[0]:.3f}, {centroid_physical_zyx[1]:.3f}, {centroid_physical_zyx[2]:.3f}) {LENGTH_UNIT}")
print(f"Bounding-box minimum (z,y,x):  {tuple(bbox_min_zyx)}")
print(f"Bounding-box maximum (z,y,x):  {tuple(bbox_max_zyx)}")
print(f"Bounding-box size in voxels:   {tuple(bbox_size_voxels_zyx)}")
print(f"Bounding-box physical size:    ({bbox_size_physical_zyx[0]:.3f}, {bbox_size_physical_zyx[1]:.3f}, {bbox_size_physical_zyx[2]:.3f}) {LENGTH_UNIT}")
print("-" * 72)
print(f"{'Plane':<8}{'Range':<16}{'Occupied':>10}{'Range length':>14}{'Maximum slice':>16}{'Maximum area':>16}")
print("-" * 80)

for plane, summary in [("XY", xy_summary), ("XZ", xz_summary), ("YZ", yz_summary)]:
    slice_range = f"{summary['first_slice']}–{summary['last_slice']}"
    print(f"{plane:<8}{slice_range:<16}{summary['occupied_slices']:>10}{summary['range_length']:>14}{summary['maximum_slice']:>16}{summary['maximum_value']:>16.3f}")

print("-" * 80)
print(f"Area unit: {LENGTH_UNIT}²")

In [ ]:
import matplotlib.pyplot as plt

profile_information = [
    ("XY", areas_xy, "z", xy_summary, voxel_size_zyx[0]),
    ("XZ", areas_xz, "y", xz_summary, voxel_size_zyx[1]),
    ("YZ", areas_yz, "x", yz_summary, voxel_size_zyx[2])
]

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, (plane, profile, coordinate_name, summary, axis_spacing) in zip(axes, profile_information):
    slice_indices = np.arange(len(profile))
    axis.plot(slice_indices, profile, color="#246BCE", linewidth=2)
    axis.fill_between(slice_indices, profile, color="#79A9E8", alpha=0.35)
    axis.axvspan(summary["first_slice"], summary["last_slice"], color="orange", alpha=0.12, label="Occupied range")
    axis.scatter(summary["maximum_slice"], summary["maximum_value"], color="red", s=55, zorder=3, label="Maximum area")
    axis.set_title(f"{plane} Cross-Sectional Area")
    axis.set_xlabel(f"{coordinate_name} slice index")
    axis.set_ylabel(f"Area ({LENGTH_UNIT}²)")
    axis.grid(alpha=0.25)
    axis.legend(fontsize=8)
    physical_span = summary["range_length"] * axis_spacing
    axis.text(0.03, 0.95, f"Occupied slices: {summary['occupied_slices']}\nPhysical span: {physical_span:.3f} {LENGTH_UNIT}", transform=axis.transAxes, va="top", bbox={"facecolor": "white", "alpha": 0.8, "edgecolor": "gray"})

plt.suptitle("Cross-Sectional Area Profiles of the Segmented Target", fontsize=14)
plt.tight_layout()
plt.show()

propagation_plane = SEED_PLANE.upper()
propagation_summary = {"XY": xy_summary, "XZ": xz_summary, "YZ": yz_summary}[propagation_plane]
print(f"Propagation plane: {propagation_plane}")
print(f"Accepted target range: {propagation_summary['first_slice']}–{propagation_summary['last_slice']}")
print(f"Occupied slices: {propagation_summary['occupied_slices']}")
print(f"Slice with maximum cross-sectional area: {propagation_summary['maximum_slice']}")
print(f"Maximum cross-sectional area: {propagation_summary['maximum_value']:.3f} {LENGTH_UNIT}²")

# Part 8 — Feature Extraction and Similar-Target Search

Part 7 quantified the segmented target using `finalized_mask_volume_3d`. In this part, the segmented region will be used as a reference template to search for structures with similar three-dimensional appearance elsewhere in the tomogram.

## 8.1 Reference Feature Extraction

The smallest 3D bounding box containing the finalized mask is extracted together with a small surrounding margin. Let the resulting image patch and mask be $I_R$ and $M_R$. Only image voxels inside the mask are used to define the reference appearance feature.

The masked reference mean and centered feature are:

$$\mu_R=\frac{1}{N_R}\sum_{\mathbf{u}}M_R(\mathbf{u})I_R(\mathbf{u}),\qquad F_R(\mathbf{u})=M_R(\mathbf{u})[I_R(\mathbf{u})-\mu_R]$$

Here, $\mathbf{u}=(z,y,x)$ is a location inside the reference patch and $N_R$ is the number of foreground voxels. Mean subtraction reduces sensitivity to local intensity offsets.

Several interpretable descriptors, including the masked intensity mean, standard deviation, percentiles, and gradient magnitude, are also reported to summarize the reference target.

## 8.2 Candidate Feature Extraction

For every valid candidate center $\mathbf{c}$, a patch with the same shape as the reference template is sampled from the tomogram. The reference mask is used as a fixed sampling footprint, producing a candidate feature from the same relative voxel locations:

$$F_{\mathbf{c}}(\mathbf{u})=M_R(\mathbf{u})[I_{\mathbf{c}}(\mathbf{u})-\mu_{\mathbf{c}}]$$

Using the same mask footprint ensures that the reference and candidate features contain the same number of elements.

## 8.3 Similarity Score

The appearance similarity between the reference and a candidate is measured using zero-mean normalized cross-correlation:

$$S(\mathbf{c})=\frac{\sum_{\mathbf{u}}F_R(\mathbf{u})F_{\mathbf{c}}(\mathbf{u})}{\sqrt{\sum_{\mathbf{u}}F_R(\mathbf{u})^2}\sqrt{\sum_{\mathbf{u}}F_{\mathbf{c}}(\mathbf{u})^2}}$$

The score normally lies between $-1$ and $1$. A value close to 1 indicates a strong match, a value near 0 indicates weak similarity, and a negative value indicates an opposing intensity pattern. Because the candidate mean and variance are calculated locally, the score is less sensitive to global brightness and contrast differences.

## 8.4 Candidate Selection

The similarity calculation produces a 3D score map. Local maxima are extracted as candidate centers, and non-maximum suppression prevents several nearby positions from representing the same structure. The original segmented target and its surrounding region are excluded so that it is not returned as its own best match.

Candidates are ranked from the highest to the lowest similarity score. `MIN_SIMILARITY` determines which candidates pass the automatic score threshold, while `TOP_K` controls how many ranked results are retained.

## 8.5 Interpretation and Limitations

A high similarity score indicates that a local region resembles the reference appearance under the selected template, but it does not prove that the candidate belongs to the same biological class. The search assumes that similar targets have approximately the same size and orientation. Strong rotations, scale changes, deformations, neighboring structures, and imaging artifacts may reduce accuracy.

The displayed reference-mask contour on a candidate patch represents the sampling footprint used by the search. It is not a newly predicted segmentation mask. Promising candidates should be inspected visually and can later be used as new prompts for segmentation and propagation.

## 8.6 Result of Part 8

The ranked search results will be stored in `similar_target_candidates`. Each record contains the candidate center, similarity score, candidate bounding box, and threshold status. The complete tutorial workflow is:

**3D tomogram → plane and slice selection → initial 2D mask → bidirectional propagation → finalized 3D mask → quantitative measurement → feature extraction → similar-target search**

## Task 8 — Search for Similar Targets

### Goal

In this task, you will extract a 3D appearance feature from the segmented target, calculate a tomogram-wide similarity map, rank local candidate matches, and inspect the highest-scoring regions.

### Your Tasks

1. Extract the reference image and mask from the finalized segmentation.
2. Inspect the reference target’s intensity and gradient descriptors.
3. Calculate the 3D normalized similarity map.
4. Exclude the original reference region from the search.
5. Apply non-maximum suppression to separate nearby candidate peaks.
6. Rank the candidates according to their similarity scores.
7. Inspect the highest-scoring candidate patches and score projections.
8. Decide which candidates are sufficiently similar for further segmentation.

### Parameters to Explore

- `REFERENCE_MARGIN`: adds surrounding context to the reference patch.
- `TOP_K`: selects the number of candidates retained.
- `MIN_SIMILARITY`: defines the minimum score for a candidate to pass.
- `MIN_PEAK_DISTANCE`: controls the separation between candidate centers.
- `EXCLUSION_MARGIN`: controls the excluded region around the original target.
- `DISPLAY_TOP_K`: controls how many candidate patches are visualized.

This search assumes that similar targets have approximately the same size and orientation as the reference. A candidate passing the score threshold must still be confirmed by visual inspection.

In [ ]:
import numpy as np

REFERENCE_MARGIN = 2  # WriteYourCodeHere: number of context voxels added around the target.

reference_volume = np.asarray(real_tomogram_display, dtype=np.float32)
reference_mask_volume = finalized_mask_volume_3d.astype(bool)

if reference_volume.ndim != 3 or reference_mask_volume.ndim != 3:
    raise ValueError("The tomogram and finalized mask must both be 3D arrays.")
if reference_volume.shape != reference_mask_volume.shape:
    raise ValueError("The tomogram and finalized mask must have the same shape.")
if not reference_mask_volume.any():
    raise ValueError("The finalized reference mask is empty.")
if REFERENCE_MARGIN < 0:
    raise ValueError("REFERENCE_MARGIN must be non-negative.")

reference_coordinates_zyx = np.argwhere(reference_mask_volume)
reference_bbox_min_zyx = reference_coordinates_zyx.min(axis=0)
reference_bbox_max_zyx = reference_coordinates_zyx.max(axis=0)
reference_start_zyx = np.maximum(reference_bbox_min_zyx - REFERENCE_MARGIN, 0)
reference_stop_zyx = np.minimum(reference_bbox_max_zyx + REFERENCE_MARGIN + 1, reference_volume.shape)
reference_slices = tuple(slice(int(start), int(stop)) for start, stop in zip(reference_start_zyx, reference_stop_zyx))

reference_image_patch = reference_volume[reference_slices].copy()
reference_mask_patch = reference_mask_volume[reference_slices].copy()

# Odd template dimensions provide an unambiguous center during correlation.
odd_padding = [(0, int(size % 2 == 0)) for size in reference_image_patch.shape]
reference_image_patch = np.pad(reference_image_patch, odd_padding, mode="edge")
reference_mask_patch = np.pad(reference_mask_patch, odd_padding, mode="constant", constant_values=False)

masked_reference_values = reference_image_patch[reference_mask_patch]

if masked_reference_values.size < 2:
    raise ValueError("The reference mask must contain at least two voxels.")
if np.std(masked_reference_values) < 1e-8:
    raise ValueError("The reference region has insufficient intensity variation for normalized matching.")

reference_intensity_mean = float(masked_reference_values.mean())
reference_intensity_std = float(masked_reference_values.std())
reference_feature_vector = (masked_reference_values - reference_intensity_mean) / reference_intensity_std

reference_centered_template = np.zeros(reference_image_patch.shape, dtype=np.float32)
reference_centered_template[reference_mask_patch] = masked_reference_values - reference_intensity_mean
reference_template_energy = float(np.sum(reference_centered_template ** 2))

gradient_components = np.gradient(reference_image_patch.astype(np.float64))
reference_gradient_magnitude = np.sqrt(sum(component ** 2 for component in gradient_components))
masked_gradient_values = reference_gradient_magnitude[reference_mask_patch]

reference_feature_summary = {
    "template_shape_zyx": tuple(reference_image_patch.shape),
    "masked_voxel_count": int(reference_mask_patch.sum()),
    "intensity_mean": reference_intensity_mean,
    "intensity_std": reference_intensity_std,
    "intensity_percentiles": tuple(np.percentile(masked_reference_values, [10, 25, 50, 75, 90])),
    "mean_gradient_magnitude": float(masked_gradient_values.mean())
}

print("Reference Target Feature")
print("-" * 66)
print(f"Reference bounding box:       {tuple(reference_bbox_min_zyx)} to {tuple(reference_bbox_max_zyx)}")
print(f"Template shape (z, y, x):    {reference_feature_summary['template_shape_zyx']}")
print(f"Masked reference voxels:     {reference_feature_summary['masked_voxel_count']}")
print(f"Intensity mean:              {reference_feature_summary['intensity_mean']:.6f}")
print(f"Intensity standard deviation:{reference_feature_summary['intensity_std']:.6f}")
print(f"Intensity percentiles:       {tuple(round(value, 6) for value in reference_feature_summary['intensity_percentiles'])}")
print(f"Mean gradient magnitude:     {reference_feature_summary['mean_gradient_magnitude']:.6f}")

In [ ]:
from scipy.ndimage import maximum_filter
from scipy.signal import fftconvolve

TOP_K = 8  # WriteYourCodeHere
MIN_SIMILARITY = 0.35  # WriteYourCodeHere
MIN_PEAK_DISTANCE = max(2, int(np.ceil(min(reference_image_patch.shape) / 3)))  # WriteYourCodeHere
EXCLUSION_MARGIN = 2  # WriteYourCodeHere

if TOP_K < 1:
    raise ValueError("TOP_K must be at least 1.")
if not (-1.0 <= MIN_SIMILARITY <= 1.0):
    raise ValueError("MIN_SIMILARITY must be between -1 and 1.")
if MIN_PEAK_DISTANCE < 1 or EXCLUSION_MARGIN < 0:
    raise ValueError("MIN_PEAK_DISTANCE must be positive and EXCLUSION_MARGIN must be non-negative.")

search_mask_kernel = reference_mask_patch.astype(np.float32)
reversed_mask_kernel = np.flip(search_mask_kernel)
reversed_template_kernel = np.flip(reference_centered_template)
masked_voxel_count = float(search_mask_kernel.sum())

local_sum = fftconvolve(reference_volume, reversed_mask_kernel, mode="same")
local_squared_sum = fftconvolve(reference_volume ** 2, reversed_mask_kernel, mode="same")
similarity_numerator = fftconvolve(reference_volume, reversed_template_kernel, mode="same")
candidate_variance_sum = np.maximum(local_squared_sum - (local_sum ** 2) / masked_voxel_count, 0.0)
similarity_denominator = np.sqrt(reference_template_energy * candidate_variance_sum)

similarity_map_3d = np.zeros(reference_volume.shape, dtype=np.float32)
stable_locations = similarity_denominator > 1e-8
similarity_map_3d[stable_locations] = similarity_numerator[stable_locations] / similarity_denominator[stable_locations]
similarity_map_3d = np.clip(similarity_map_3d, -1.0, 1.0)

template_half_size_zyx = np.asarray(reference_image_patch.shape) // 2
valid_centers = np.zeros(reference_volume.shape, dtype=bool)
valid_slices = tuple(slice(int(half), int(size - half)) for half, size in zip(template_half_size_zyx, reference_volume.shape))
valid_centers[valid_slices] = True

# Exclude every candidate whose template would overlap the original target and exclusion margin.
exclusion_start_zyx = np.maximum(reference_bbox_min_zyx - template_half_size_zyx - EXCLUSION_MARGIN, 0)
exclusion_stop_zyx = np.minimum(reference_bbox_max_zyx + template_half_size_zyx + EXCLUSION_MARGIN + 1, reference_volume.shape)
exclusion_slices = tuple(slice(int(start), int(stop)) for start, stop in zip(exclusion_start_zyx, exclusion_stop_zyx))
valid_centers[exclusion_slices] = False

search_similarity_map_3d = np.where(valid_centers, similarity_map_3d, -np.inf)
peak_filter_size = 2 * MIN_PEAK_DISTANCE + 1
local_maximum_map = maximum_filter(search_similarity_map_3d, size=peak_filter_size, mode="constant", cval=-np.inf)
peak_coordinates_zyx = np.argwhere(np.isfinite(search_similarity_map_3d) & (search_similarity_map_3d == local_maximum_map))

if len(peak_coordinates_zyx) == 0:
    raise ValueError("No valid candidate peaks were found. Reduce the template size or exclusion region.")

peak_scores = search_similarity_map_3d[tuple(peak_coordinates_zyx.T)]
ranked_order = np.argsort(peak_scores)[::-1][:TOP_K]
ranked_coordinates_zyx = peak_coordinates_zyx[ranked_order]
ranked_scores = peak_scores[ranked_order]

similar_target_candidates = []

for rank, (center_zyx, score) in enumerate(zip(ranked_coordinates_zyx, ranked_scores), start=1):
    candidate_bbox_min = center_zyx - template_half_size_zyx
    candidate_bbox_max = center_zyx + template_half_size_zyx
    similar_target_candidates.append({
        "rank": rank,
        "center_zyx": tuple(int(value) for value in center_zyx),
        "similarity": float(score),
        "passes_threshold": bool(score >= MIN_SIMILARITY),
        "bbox_min_zyx": tuple(int(value) for value in candidate_bbox_min),
        "bbox_max_zyx": tuple(int(value) for value in candidate_bbox_max)
    })

print("Ranked Similar-Target Candidates")
print("-" * 88)
print(f"{'Rank':>5} {'Center (z, y, x)':>22} {'Similarity':>12} {'Pass':>8} {'Candidate bounding box':>34}")
print("-" * 88)

for candidate in similar_target_candidates:
    candidate_box = f"{candidate['bbox_min_zyx']} to {candidate['bbox_max_zyx']}"
    print(f"{candidate['rank']:>5} {str(candidate['center_zyx']):>22} {candidate['similarity']:>12.4f} {str(candidate['passes_threshold']):>8} {candidate_box:>34}")

print("-" * 88)
print(f"Candidates passing MIN_SIMILARITY = {MIN_SIMILARITY:.2f}: {sum(candidate['passes_threshold'] for candidate in similar_target_candidates)}")
print("Passing the threshold indicates appearance similarity, not confirmed target identity.")

In [ ]:
import matplotlib.pyplot as plt

DISPLAY_TOP_K = min(5, len(similar_target_candidates))  # WriteYourCodeHere
display_candidates = similar_target_candidates[:DISPLAY_TOP_K]
central_template_z = reference_image_patch.shape[0] // 2

candidate_image_patches = []

for candidate in display_candidates:
    candidate_min = np.asarray(candidate["bbox_min_zyx"])
    candidate_max = np.asarray(candidate["bbox_max_zyx"]) + 1
    candidate_slices = tuple(slice(int(start), int(stop)) for start, stop in zip(candidate_min, candidate_max))
    candidate_image_patches.append(reference_volume[candidate_slices])

display_values = np.concatenate([reference_image_patch.ravel()] + [patch.ravel() for patch in candidate_image_patches])
display_min, display_max = np.percentile(display_values, [1, 99])

fig, axes = plt.subplots(1, DISPLAY_TOP_K + 1, figsize=(3.3 * (DISPLAY_TOP_K + 1), 3.5), squeeze=False)
axes = axes[0]

axes[0].imshow(reference_image_patch[central_template_z], cmap="gray", origin="lower", vmin=display_min, vmax=display_max)
if reference_mask_patch[central_template_z].any():
    axes[0].contour(reference_mask_patch[central_template_z], levels=[0.5], colors="yellow", linewidths=1.2, origin="lower")
axes[0].set_title("Reference Target")

for axis, candidate, patch in zip(axes[1:], display_candidates, candidate_image_patches):
    axis.imshow(patch[central_template_z], cmap="gray", origin="lower", vmin=display_min, vmax=display_max)
    if reference_mask_patch[central_template_z].any():
        axis.contour(reference_mask_patch[central_template_z], levels=[0.5], colors="cyan", linewidths=1.0, origin="lower")
    status = "Pass" if candidate["passes_threshold"] else "Below threshold"
    axis.set_title(f"Rank {candidate['rank']}: {candidate['similarity']:.3f}\n{candidate['center_zyx']} | {status}", fontsize=9)

for axis in axes:
    axis.set_xlabel("x within patch")
    axis.set_ylabel("y within patch")

plt.suptitle("Reference and Ranked Candidate Patches\nCandidate contours show the reference sampling footprint, not a new segmentation", fontsize=13)
plt.tight_layout()
plt.show()

In [ ]:
finite_scores = search_similarity_map_3d[np.isfinite(search_similarity_map_3d)]

if finite_scores.size == 0:
    raise ValueError("The similarity map contains no valid search positions.")

projection_floor = float(finite_scores.min())
display_similarity_map = np.where(np.isfinite(search_similarity_map_3d), search_similarity_map_3d, projection_floor)
similarity_projections = [
    display_similarity_map.max(axis=0),
    display_similarity_map.max(axis=1),
    display_similarity_map.max(axis=2)
]
projection_titles = ["XY Similarity Projection", "XZ Similarity Projection", "YZ Similarity Projection"]
projection_labels = [("x", "y"), ("x", "z"), ("y", "z")]
reference_center_zyx = np.rint(reference_coordinates_zyx.mean(axis=0)).astype(int)

fig, axes = plt.subplots(1, 3, figsize=(15, 4.5))

for axis, projection, title, labels in zip(axes, similarity_projections, projection_titles, projection_labels):
    image_handle = axis.imshow(projection, cmap="viridis", origin="lower", vmin=max(-0.2, projection_floor), vmax=1.0)
    axis.set_title(title)
    axis.set_xlabel(labels[0])
    axis.set_ylabel(labels[1])
    plt.colorbar(image_handle, ax=axis, fraction=0.046, pad=0.04)

axes[0].scatter(reference_center_zyx[2], reference_center_zyx[1], marker="*", s=150, c="white", edgecolors="black", label="Reference")
axes[1].scatter(reference_center_zyx[2], reference_center_zyx[0], marker="*", s=150, c="white", edgecolors="black", label="Reference")
axes[2].scatter(reference_center_zyx[1], reference_center_zyx[0], marker="*", s=150, c="white", edgecolors="black", label="Reference")

for candidate in display_candidates:
    z, y, x = candidate["center_zyx"]
    color = "red" if candidate["passes_threshold"] else "orange"
    axes[0].scatter(x, y, s=45, facecolors="none", edgecolors=color, linewidths=1.5)
    axes[1].scatter(x, z, s=45, facecolors="none", edgecolors=color, linewidths=1.5)
    axes[2].scatter(y, z, s=45, facecolors="none", edgecolors=color, linewidths=1.5)

for axis in axes:
    axis.legend(loc="upper right", fontsize=8)

plt.suptitle("Maximum Projections of the 3D Similarity Map", fontsize=14)
plt.tight_layout()
plt.show()

print("Red circles: candidates passing the similarity threshold.")
print("Orange circles: ranked candidates below the threshold.")
print("The white star marks the original segmented target.")

# Part 9 — Summary and Outlook

This tutorial developed a complete educational workflow for interactive CryoET tomogram segmentation. Starting from the basic representation of a 3D volume, the notebook demonstrated how one user-selected target on a 2D slice can be expanded into a 3D segmentation, inspected and corrected, converted into quantitative measurements, and finally used as a reference for finding similar structures.

## 9.1 What Was Covered

Part 1 introduced the relationship between tomograms, voxels, viewing planes, 2D masks, and 3D masks using a synthetic volume with known ground truth. Part 2 then moved to a real CryoET tomogram crop and discussed data loading, axis conventions, percentile normalization, physical voxel spacing, annotations, and imaging challenges such as noise and the missing-wedge effect.

In Part 3, a seed plane and slice were selected, and positive and optional negative point prompts were used to create an initial 2D mask. Part 4 converted the current mask into automatically generated foreground and background prompts that could be transferred to a neighboring slice. Part 5 repeated this process in both directions and applied continuity checks, including mask area, area ratio, adjacent-mask IoU, and centroid displacement, to reduce propagation drift.

Part 6 inserted the accepted 2D masks into their correct tomogram coordinates to reconstruct a 3D binary mask. Orthogonal views, mask overlays, projections, connected components, and local correction were then used to inspect and finalize the segmentation. Part 7 converted the finalized mask into measurements such as foreground voxel count, physical volume, centroid, bounding box, occupied slice range, and cross-sectional area profiles.

Finally, Part 8 treated the segmented target as a 3D reference template. Masked intensity and gradient information were extracted, normalized cross-correlation was calculated throughout the tomogram, and local similarity peaks were ranked as possible locations of additional targets.

## 9.2 Complete Workflow

The complete workflow developed in this notebook is:

**Synthetic volume exploration → real tomogram loading → plane and seed-slice selection → user prompt → initial 2D mask → automatic prompt generation → bidirectional propagation → continuity checks → 3D mask reconstruction → multi-plane inspection and correction → quantitative measurement → reference-feature extraction → similar-target search**

The main outputs produced throughout the workflow include:

| Output | Meaning |
|---|---|
| `initial_2d_mask` | Segmentation of the user-selected target on the seed slice |
| `propagated_masks` | Accepted 2D masks indexed by their slice positions |
| `finalized_mask_volume_3d` | Inspected and corrected 3D segmentation |
| `measurement_summary` | Size, position, bounding-box, and slice-span measurements |
| `area_profiles` | Cross-sectional area changes along the three spatial axes |
| `similar_target_candidates` | Ranked locations with appearances similar to the reference target |

## 9.3 Main Lessons

The first important lesson is that a 3D segmentation is not produced by simply extending one 2D mask through the volume. The target changes across slices, so each neighboring mask must be predicted from the local image information and placed back into the correct 3D coordinate system.

The second lesson is that interactive segmentation combines human guidance with automatic processing. The user identifies the intended structure and can correct inaccurate results, while automatic prompt generation and propagation reduce the need to draw the target manually on every slice.

The third lesson is that spatial continuity is useful but does not guarantee segmentation accuracy. A mask may pass the area, overlap, and centroid-displacement checks while still including an incorrect structure. Therefore, automatic stopping criteria must be combined with orthogonal inspection, tomogram–mask overlays, and local correction.

The final lesson is that segmentation, measurement, and target search are related but distinct stages. Measurements describe the current mask and are only as reliable as the segmentation from which they were calculated. Similarly, a high similarity score identifies a possible matching region but does not confirm its biological identity or provide a new segmentation mask.

## 9.4 Limitations of the Educational Implementation

This notebook is a lightweight educational adaptation of ideas from promptable segmentation and CryoSAM rather than a complete reproduction of the original systems. The initial mask and propagated masks are generated using simplified image-processing procedures, while a full research implementation may use pretrained foundation-model image features, mask decoders, cross-plane self-prompting, and more advanced hierarchical feature matching.

Real CryoET data also contain weak boundaries, low signal-to-noise ratios, anisotropic resolution, missing-wedge artifacts, and nearby structures with similar appearances. The propagation result can therefore be sensitive to the selected plane, seed slice, prompt positions, threshold values, and stopping criteria.

The similar-target search assumes that candidate structures have approximately the same size, orientation, and appearance as the reference target. Rotated, deformed, partially visible, or differently scaled targets may receive lower similarity scores. Every candidate must therefore be treated as a hypothesis that requires further inspection and, when appropriate, a new round of prompt-based segmentation.

## 9.5 Possible Extensions

The workflow could be extended by replacing the educational 2D segmentation method with SAM, CryoSAM, or another promptable foundation model. Propagation could also combine predictions from multiple viewing planes, use uncertainty estimates to identify slices requiring correction, or restart propagation from a corrected intermediate slice.

If expert voxel-level annotations are available, the finalized mask could be evaluated using Dice score, IoU, boundary distance, centroid error, and volume error. The similar-target search could also be improved using learned 3D embeddings, rotation-aware matching, multi-scale templates, or clustering of candidate features.

For larger tomograms, the workflow could be adapted to process overlapping 3D blocks, use GPU acceleration, and export masks in formats compatible with CryoET visualization and analysis software.

## 9.6 Final Conclusion

This tutorial showed how a small amount of user input can guide a complete 3D CryoET analysis pipeline. A single prompt on an informative slice was converted into an initial mask, propagated across neighboring slices, reconstructed and validated as a 3D segmentation, translated into quantitative measurements, and used to search for similar structures.

The central idea is not to remove the researcher from the workflow, but to combine human target selection and quality control with automatic propagation and analysis. This interaction provides a practical foundation for developing more advanced, efficient, and reliable CryoET segmentation systems.

---

**End of Tutorial — Interactive CryoET Tomogram Segmentation with Prompt Propagation**